In [ ]:
# Create directory structure for the project
import os

# Define the directory structure
directories = [
    'configs',
    'data',
    'environment',
    'models',
    'agents',
    'utils',
    'scripts'
]

# Create directories
for directory in directories:
    os.makedirs(directory, exist_ok=True)
    print(f"Created directory: {directory}")

print("Directory structure created successfully!")

In [ ]:
%%writefile configs/config.py
"""
Centralized configuration for the Multi-Agent DRL Inventory Management System
All hyperparameters and system constants are defined here.
"""

class EnvironmentConfig:
    """Configuration for the inventory management environment"""
    def __init__(self):
        # Core environment parameters
        self.NUM_PRODUCTS = 5    # Number of distinct finished goods (M)
        self.FG_LEAD_TIME = 3    # Production lead time for finished goods (days)
        self.RM_LEAD_TIME = 5    # Procurement lead time for raw materials (days)

        # Cost parameters
        self.COST_FG_HOLDING = [0.1] * 5    # Per-unit, per-day holding cost for each FG
        self.COST_FG_SHORTAGE = [10.0] * 5    # Per-unit shortage cost for each FG (lost sales)
        self.COST_RM_HOLDING = 0.02    # Per-unit, per-day holding cost for raw material
        self.COST_RM_ORDER = 0.5    # Fixed cost for placing raw material order

        # Demand and forecasting
        self.FORECAST_HORIZON = 7    # Days included in demand forecast
        self.INITIAL_INVENTORY_FG = [80] * 5    # Starting inventory for finished goods
        self.INITIAL_INVENTORY_RM = 1500    # Starting inventory for raw materials

        # Demand simulation parameters (for synthetic data generation)
        self.DEMAND_MEAN = [5.0] * 5    # Mean demand for each product
        self.DEMAND_STD = [2.0] * 5    # Standard deviation for demand
        self.SEASONALITY_AMPLITUDE = 2.0    # Amplitude of seasonal patterns
        self.MAX_DAILY_DEMAND = 15    # Maximum possible daily demand per product

class ModelConfig:
    """Configuration for neural network architectures"""
    def __init__(self):
        # CNN Time-Series Encoder
        self.HIDDEN_DIM = 32    # Neurons in hidden layers
        self.CNN_OUT_CHANNELS = 16    # Output channels for CNN encoder
        self.CNN_KERNEL_SIZE = 2    # Kernel size for causal convolutions
        self.CNN_NUM_LAYERS = 4    # Number of dilated convolutional layers

        # Transformer (for Phase 3)
        self.TRANSFORMER_D_MODEL = 128    # Hidden dimension for transformer
        self.TRANSFORMER_NHEAD = 8    # Number of attention heads
        self.TRANSFORMER_NUM_LAYERS = 4    # Number of transformer layers
        self.TRANSFORMER_DROPOUT = 0.1    # Dropout rate

        # Optimization
        self.LEARNING_RATE_ACTOR = 3e-4    # Learning rate for actor network
        self.LEARNING_RATE_CRITIC = 1e-3    # Learning rate for critic network
        self.OPTIMIZER_EPS = 1e-5    # Epsilon for Adam optimizer

class PPOConfig:
    """Configuration for PPO algorithm"""
    def __init__(self):
        # Advantage estimation
        self.GAMMA = 0.99    # Discount factor
        self.LAMBDA_GAE = 0.95    # GAE parameter

        # Clipping and optimization
        self.EPSILON_CLIP = 0.2    # PPO clipping parameter
        self.PPO_EPOCHS = 10    # Optimization epochs per update
        self.MINIBATCH_SIZE = 64    # Minibatch size
        self.ENTROPY_COEFF = 0.01    # Entropy bonus coefficient

        # Value function
        self.VALUE_COEFF = 0.5    # Value loss coefficient
        self.MAX_GRAD_NORM = 0.5    # Gradient clipping

class TrainingConfig:
    """Configuration for training process"""
    def __init__(self):
        # Main training
        self.TOTAL_TIMESTEPS = 200000    # Reduced for Colab testing
        self.UPDATE_TIMESTEPS = 2048    # Steps between PPO updates
        self.EVAL_FREQ = 10000    # Evaluation frequency
        self.SAVE_FREQ = 50000    # Model saving frequency

        # Imitation learning
        self.IMITATION_EPOCHS = 50    # Epochs for behavioral cloning
        self.IMITATION_BATCH_SIZE = 32    # Batch size for imitation learning
        self.EXPERT_DATASET_SIZE = 5000    # Size of expert dataset

        # Regularization
        self.LAMBDA_MONO = 0.1    # Monotonicity regularization weight - CRITICAL ADDITION

        # Base-stock levels for heuristic policy
        self.BASE_STOCK_FG = [20] * 5    # Base-stock levels for finished goods
        self.BASE_STOCK_RM = 30    # Base-stock level for raw material

class ColabConfig:
    """Colab-specific configuration"""
    def __init__(self):
        self.CHECKPOINT_DIR = "/content/checkpoints"
        self.TENSORBOARD_DIR = "/content/tensorboard"
        self.USE_GPU = True
        self.DEBUG_MODE = True

# Create global config object
class Config:
    def __init__(self):
        self.env = EnvironmentConfig()
        self.model = ModelConfig()
        self.ppo = PPOConfig()
        self.training = TrainingConfig()
        self.colab = ColabConfig()

    def print_config(self):
        """Print the entire configuration for verification"""
        print("== Configuration Summary ==")
        print(f"Environment: {self.env.NUM_PRODUCTS} products, FG lead time: {self.env.FG_LEAD_TIME}, RM lead time: {self.env.RM_LEAD_TIME}")
        print(f"Model: Hidden dim: {self.model.HIDDEN_DIM}, Learning rates: {self.model.LEARNING_RATE_ACTOR}/{self.model.LEARNING_RATE_CRITIC}")
        print(f"PPO: Gamma: {self.ppo.GAMMA}, Clip: {self.ppo.EPSILON_CLIP}, Epochs: {self.ppo.PPO_EPOCHS}")
        print(f"Training: Total steps: {self.training.TOTAL_TIMESTEPS}, Update freq: {self.training.UPDATE_TIMESTEPS}")
        print(f"Monotonicity Regularization: Lambda: {self.training.LAMBDA_MONO}")
        print(f"Colab: GPU: {self.colab.USE_GPU}, Debug: {self.colab.DEBUG_MODE}")

# Global config instance
config = Config()

In [ ]:
%%writefile utils/helpers.py
"""
Utility functions for the inventory management project
"""

import numpy as np
import torch
import matplotlib.pyplot as plt
from typing import Dict, List
from torch.utils.tensorboard import SummaryWriter  # Correct import location

def normalize_array(arr: np.ndarray) -> np.ndarray:
    """Normalize numpy array to [0, 1] range"""
    if np.max(arr) == np.min(arr):
        return np.zeros_like(arr)
    return (arr - np.min(arr)) / (np.max(arr) - np.min(arr))

def denormalize_array(arr: np.ndarray, original_min: float, original_max: float) -> np.ndarray:
    """Denormalize array back to original range"""
    return arr * (original_max - original_min) + original_min

def calculate_moving_average(data: List[float], window_size: int) -> List[float]:
    """Calculate moving average of a time series"""
    if len(data) < window_size:
        return data
    return [np.mean(data[i:i+window_size]) for i in range(len(data) - window_size + 1)]

def plot_training_curves(rewards: List[float], losses: List[float], title: str = "Training Progress"):
    """Plot training rewards and losses"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    # Plot rewards
    ax1.plot(rewards)
    ax1.set_title(f'{title} - Rewards')
    ax1.set_xlabel('Episode')
    ax1.set_ylabel('Total Reward')
    ax1.grid(True)

    # Plot losses
    ax2.plot(losses)
    ax2.set_title(f'{title} - Losses')
    ax2.set_xlabel('Update Step')
    ax2.set_ylabel('Loss')
    ax2.grid(True)

    plt.tight_layout()
    plt.show()

def dict_to_tensor(state_dict: Dict, device: torch.device) -> torch.Tensor:
    """Convert state dictionary to flattened tensor"""
    tensors = []
    for key in sorted(state_dict.keys()):
        value = state_dict[key]
        if isinstance(value, (int, float)):
            value = [value]
        tensors.append(torch.tensor(value, dtype=torch.float32))

    return torch.cat(tensors).to(device)

class Logger:
    """Simple logger for training progress"""
    def __init__(self, log_dir: str = "/content/logs"):
        self.writer = SummaryWriter(log_dir)
        self.episode_rewards = []

    def log_episode(self, episode: int, reward: float, length: int):
        """Log episode statistics"""
        self.episode_rewards.append(reward)
        self.writer.add_scalar('Training/Episode_Reward', reward, episode)
        self.writer.add_scalar('Training/Episode_Length', length, episode)

    def log_losses(self, step: int, policy_loss: float, value_loss: float, entropy_loss: float):
        """Log training losses"""
        self.writer.add_scalar('Losses/Policy_Loss', policy_loss, step)
        self.writer.add_scalar('Losses/Value_Loss', value_loss, step)
        self.writer.add_scalar('Losses/Entropy_Loss', entropy_loss, step)

    def close(self):
        """Close the logger"""
        self.writer.close()

# Test the helpers
if __name__ == "__main__":
    # Test normalization
    test_data = np.array([1, 2, 3, 4, 5])
    normalized = normalize_array(test_data)
    print(f"Original: {test_data}")
    print(f"Normalized: {normalized}")

    # Test moving average
    moving_avg = calculate_moving_average(list(test_data), 3)
    print(f"Moving average (window=3): {moving_avg}")

In [ ]:
%%writefile environment/inventory_env.py
"""

Inventory Management Environment with CRITICAL FIXES:
1. Unified shared reward signal for both agents
2. Simplified cost function without ad-hoc penalties
"""

import numpy as np
import gym
from gym import spaces
from typing import Dict, Tuple, Optional, List
import configs.config as config

class InventoryMAMDPEnv(gym.Env):
    """
    VERSION: Environment with unified shared reward and simplified cost function
    """

    def __init__(self, config_obj=None):
        super(InventoryMAMDPEnv, self).__init__()

        # Use provided config or default to global config
        self.cfg = config_obj if config_obj else config.config.env

        # Store core parameters
        self.num_products = self.cfg.NUM_PRODUCTS
        self.fg_lead_time = self.cfg.FG_LEAD_TIME
        self.rm_lead_time = self.cfg.RM_LEAD_TIME

        # Cost parameters
        self.cost_fg_holding = np.array(self.cfg.COST_FG_HOLDING)
        self.cost_fg_shortage = np.array(self.cfg.COST_FG_SHORTAGE)
        self.cost_rm_holding = self.cfg.COST_RM_HOLDING
        self.cost_rm_order = self.cfg.COST_RM_ORDER

        # Demand simulation parameters
        self.demand_mean = np.array(self.cfg.DEMAND_MEAN)
        self.demand_std = np.array(self.cfg.DEMAND_STD)
        self.seasonality_amplitude = self.cfg.SEASONALITY_AMPLITUDE

        # Define action spaces for both agents
        self.action_space_fg = spaces.Box(
            low=0,
            high=np.inf,
            shape=(self.num_products,),
            dtype=np.float32
        )
        self.action_space_rm = spaces.Box(
            low=0,
            high=np.inf,
            shape=(1,),
            dtype=np.float32
        )

        # Define observation spaces - UPDATED for FG agent
        self.observation_space = spaces.Dict({
            'fg_agent': self._get_fg_observation_space(),
            'rm_agent': self._get_rm_observation_space()
        })

        # Initial conditions
        self.current_step = 0
        self.fg_inventory = np.array(self.cfg.INITIAL_INVENTORY_FG, dtype=np.float32)
        self.rm_inventory = self.cfg.INITIAL_INVENTORY_RM
        self.fg_pipeline = np.zeros((self.num_products, self.fg_lead_time), dtype=np.float32)
        self.rm_pipeline = np.full(self.rm_lead_time, 15.0, dtype=np.float32)
        self.demand_history = []
        self.max_steps = 365
        self.rm_stockout_count = 0

        if config.config.colab.DEBUG_MODE:
            print("Inventory Environment initialized successfully")
            print(f"Products: {self.num_products}, FG Lead Time: {self.fg_lead_time}, RM Lead Time: {self.rm_lead_time}")

    def _get_fg_observation_space(self) -> spaces.Box:
        """Enhanced FG observation space"""
        # New calculation: 5 + 15 + 4 + 3 + 4 + 2 + 4 = 37 dimensions
        obs_dim = (self.num_products +                    # current FG inventory (5)
                  self.num_products * self.fg_lead_time + # FG pipeline (15)
                  4 +                                    # demand metrics (4)
                  3 +                                    # historical patterns (3)
                  4 +                                    # production metrics (4)
                  2 +                                    # temporal features (2)
                  4)                                     # RM info (4)
        return spaces.Box(low=-np.inf, high=np.inf, shape=(obs_dim,), dtype=np.float32)

    def _get_rm_observation_space(self) -> spaces.Box:
        # New dimension: 1 + 5 + 4 + 4 + 3 + 3 = 20
        obs_dim = (1 +                    # current RM inventory
                  self.rm_lead_time +     # RM pipeline (5)
                  4 +                     # production metrics
                  4 +                     # demand metrics
                  3 +                     # historical patterns
                  3)                      # system state
        return spaces.Box(low=-np.inf, high=np.inf, shape=(obs_dim,), dtype=np.float32)

    def reset(self) -> Dict[str, np.ndarray]:
        """Reset the environment with initial conditions"""
        self.fg_inventory = np.array(self.cfg.INITIAL_INVENTORY_FG, dtype=np.float32)
        self.rm_inventory = self.cfg.INITIAL_INVENTORY_RM
        self.fg_pipeline = np.zeros((self.num_products, self.fg_lead_time), dtype=np.float32)
        self.rm_pipeline = np.full(self.rm_lead_time, 15.0, dtype=np.float32)
        self.current_step = 0
        self.rm_stockout_count = 0

        # Initialize demand history for forecasting
        self.demand_history = []
        for _ in range(self.cfg.FORECAST_HORIZON):
            self.demand_history.append(self._simulate_demand())

        return self._get_obs()

    def _get_fg_observation(self) -> np.ndarray:
        """Enhanced FG observation with predictive features"""
        obs_components = []

        # 1. Current FG inventory (M values) - EXISTING
        obs_components.append(self.fg_inventory)

        # 2. FG pipeline flattened (M * FG_LEAD_TIME values) - EXISTING
        obs_components.append(self.fg_pipeline.flatten())

        # 3. Enhanced demand forecasting (NEW - like RM agent)
        forecast = self._generate_demand_forecast()
        if forecast.size > 0:
            total_forecast = np.sum(forecast, axis=0)
            demand_metrics = [
                np.max(total_forecast),           # Peak demand forecast
                np.mean(total_forecast),          # Average demand
                np.std(total_forecast),           # Demand variability
                total_forecast[0]                 # Immediate demand
            ]
        else:
            demand_metrics = [0, 0, 0, 0]
        obs_components.append(demand_metrics)

        # 4. Historical patterns (NEW - like RM agent)
        if len(self.demand_history) > 7:
            recent_demand = np.array(self.demand_history[-7:])
            total_recent = np.sum(recent_demand, axis=1)
            historical_metrics = [
                np.max(total_recent),
                np.mean(total_recent),
                np.std(total_recent)
            ]
        else:
            historical_metrics = [0, 0, 0]
        obs_components.append(historical_metrics)

        # 5. Production efficiency metrics (NEW)
        production_metrics = [
            np.sum(self.fg_pipeline),             # Total planned production
            np.max(self.fg_pipeline),             # Peak production
            np.std(self.fg_pipeline),             # Production variability
            self.rm_inventory / (np.sum(self.fg_pipeline) + 1e-6)  # RM coverage ratio
        ]
        obs_components.append(production_metrics)

        # 6. Temporal features (EXISTING)
        day_of_year = self.current_step % 365
        temporal_features = [
            np.sin(2 * np.pi * day_of_year / 365),
            np.cos(2 * np.pi * day_of_year / 365)
        ]
        obs_components.append(temporal_features)

        # 7. RM inventory information (EXISTING but enhanced)
        rm_info = [
            self.rm_inventory,                    # Current RM Inventory level
            np.sum(self.rm_pipeline),             # Total RM in pipeline
            self.rm_stockout_count,               # Historical reliability
            np.mean(self.rm_pipeline)             # Average pipeline flow
        ]
        obs_components.append(rm_info)

        return np.concatenate([arr.flatten() if hasattr(arr, 'flatten') else np.array(arr)
                              for arr in obs_components], dtype=np.float32)

    def _get_rm_observation(self) -> np.ndarray:
        """Enhanced RM observation with predictive features"""
        obs_components = []

        # 1. Current RM status (existing)
        obs_components.append([self.rm_inventory])
        obs_components.append(self.rm_pipeline)

        # 2. Enhanced production signals (NEW)
        production_metrics = [
            np.sum(self.fg_pipeline[:, -1]),  # Immediate production needs
            np.sum(self.fg_pipeline),         # Total pipeline production
            np.max(self.fg_pipeline),         # Peak production demand
            np.std(self.fg_pipeline)          # Production variability
        ]
        obs_components.append(production_metrics)

        # 3. Enhanced demand forecasting (NEW)
        forecast = self._generate_demand_forecast()
        if forecast.size > 0:
            total_forecast = np.sum(forecast, axis=0)
            demand_metrics = [
                np.max(total_forecast),       # Peak demand forecast
                np.mean(total_forecast),      # Average demand
                np.std(total_forecast),       # Demand variability
                total_forecast[0]             # Immediate demand
            ]
        else:
            demand_metrics = [0, 0, 0, 0]
        obs_components.append(demand_metrics)

        # 4. Historical patterns (NEW)
        if len(self.demand_history) > 7:
            recent_demand = np.array(self.demand_history[-7:])
            total_recent = np.sum(recent_demand, axis=1)
            historical_metrics = [
                np.max(total_recent),
                np.mean(total_recent),
                np.std(total_recent)
            ]
        else:
            historical_metrics = [0, 0, 0]
        obs_components.append(historical_metrics)

        # 5. System state awareness (NEW)
        system_state = [
            self.current_step / 365.0,        # Training progress
            min(self.rm_stockout_count, 10),  # Historical failures (capped)
            np.sum(self.rm_pipeline > 0) / len(self.rm_pipeline)  # Pipeline utilization
        ]
        obs_components.append(system_state)

        return np.concatenate([arr.flatten() if hasattr(arr, 'flatten') else np.array(arr)
                              for arr in obs_components], dtype=np.float32)

    def step(self, joint_action: Dict[str, np.ndarray]) -> Tuple[Dict[str, np.ndarray], float, bool, Dict]:
        """Execute one timestep with CRITICAL FIX: unified shared reward"""

        # Extract actions
        fg_action = joint_action['fg_agent']
        rm_action = joint_action['rm_agent'][0]

        # Clip actions to be non-negative
        fg_action = np.clip(fg_action, 0, None)
        rm_action = max(0, rm_action)

        # Action clipping for FG agent
        max_reasonable_production = self.rm_inventory * 1.5
        current_total_planned = np.sum(fg_action)

        if current_total_planned > max_reasonable_production:
            scale_factor = max_reasonable_production / current_total_planned
            fg_action = fg_action * scale_factor

        # Cap individual product orders
        max_per_product = 25.0
        fg_action = np.clip(fg_action, 0, max_per_product)

        # 1. Fulfill demand and update FG inventory
        realized_demand = self._simulate_demand()
        lost_sales, fulfilled_demand = self._fulfill_demand(realized_demand)

        # 2. Update FG production pipeline (FIFO)
        self._update_fg_pipeline(fg_action)

        # 3. Update RM inventory and handle production constraints
        actual_production = self._update_rm_inventory(fg_action)

        # 4. Update RM procurement pipeline (FIFO)
        self._update_rm_pipeline(rm_action)

        # 6. Unified shared reward for both agents
        (shared_reward, fg_holding_cost, fg_shortage_cost, rm_holding_cost, rm_order_cost, rm_stockout_penalty) = self._calculate_total_reward_simplified(fulfilled_demand, lost_sales, rm_action)

        # 5. total system cost - SIMPLIFIED VERSION
        total_cost = fg_holding_cost + rm_holding_cost + rm_order_cost

        # 7. Update demand history for forecasting
        self.demand_history.append(realized_demand)
        if len(self.demand_history) > self.cfg.FORECAST_HORIZON:
            self.demand_history.pop(0)

        # 8. Increment step and check termination
        self.current_step += 1
        done = self.current_step >= self.max_steps

        # 9. Get new observations
        obs = self._get_obs()

        # 10. Prepare info dictionary
        info = {
            'step': self.current_step,
            'total_cost': total_cost,
            'fg_inventory': self.fg_inventory.copy(),
            'rm_inventory': self.rm_inventory,
            'lost_sales': lost_sales,
            'fulfilled_demand': fulfilled_demand,
            'rm_stockout_count': self.rm_stockout_count,
            'actual_production': actual_production,
            'planned_production': fg_action,
            # ADD cost components for detailed analysis
            'fg_holding_cost': fg_holding_cost,
            'fg_shortage_cost': fg_shortage_cost,
            'rm_holding_cost': rm_holding_cost,
            'rm_stockout_penalty': rm_stockout_penalty,
            'rm_order_cost': rm_order_cost
        }

        # CRITICAL FIX: Return same reward for both agents
        rewards = {
            'fg_agent': shared_reward,
            'rm_agent': shared_reward
        }

        return obs, rewards, done, info

    def _calculate_total_reward_simplified(self, fulfilled_demand: np.ndarray, lost_sales: np.ndarray, rm_action: float) -> float:
        """Enhanced cost function with CRITICAL RM stockout penalty"""

        # FG costs
        fg_holding_cost = np.sum(self.cost_fg_holding * self.fg_inventory)
        fg_shortage_cost = np.sum(self.cost_fg_shortage * lost_sales)

        # RM costs - CRITICAL: Massive penalty for stockouts
        rm_holding_cost = self.cost_rm_holding * self.rm_inventory
        rm_order_cost = self.cost_rm_order if rm_action > 0 else 0

        # FIXED: Smooth quadratic penalty for RM stockouts
        rm_stockout_penalty = 0
        planned_production = np.sum(self.fg_pipeline[:, -1]) # New production planned
        if planned_production > 0:
            # Ultra-smooth penalty with linear-quadratic combination
            rm_shortage = max(0, planned_production - self.rm_inventory)
            if rm_shortage > 0:
                # Combined linear + quadratic for very smooth gradients
                rm_stockout_penalty = 1.0 * rm_shortage + 0.5 * (rm_shortage ** 2)
            else:
                rm_stockout_penalty = 0

        total_reward = -(fg_holding_cost + fg_shortage_cost +
                      rm_holding_cost + rm_order_cost + rm_stockout_penalty)

        return (total_reward, fg_holding_cost, fg_shortage_cost, rm_holding_cost, rm_order_cost, rm_stockout_penalty)

    def _simulate_demand(self) -> np.ndarray:
        """Simulate daily demand for each product"""
        base_demand = np.random.normal(self.demand_mean, self.demand_std)
        base_demand = np.clip(base_demand, 0, self.cfg.MAX_DAILY_DEMAND)

        day_of_week = self.current_step % 7
        seasonality_factor = 1.0 + self.seasonality_amplitude * np.sin(2 * np.pi * day_of_week / 7)

        demand = base_demand * seasonality_factor
        return np.clip(demand, 0, None).astype(np.float32)

    def _fulfill_demand(self, demand: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        """Fulfill customer demand and calculate lost sales"""
        fulfilled = np.minimum(self.fg_inventory, demand)
        lost_sales = demand - fulfilled
        self.fg_inventory -= fulfilled
        self.fg_inventory = np.clip(self.fg_inventory, 0, None)
        return lost_sales, fulfilled

    def _update_fg_pipeline(self, new_production: np.ndarray):
        """Update finished goods pipeline with FIFO logic"""
        for d in range(self.fg_lead_time - 1):
            self.fg_pipeline[:, d] = self.fg_pipeline[:, d + 1]
        self.fg_pipeline[:, -1] = new_production
        completed_production = self.fg_pipeline[:, 0].copy()
        self.fg_inventory += completed_production
        self.fg_pipeline[:, 0] = 0

    def _update_rm_inventory(self, planned_production: np.ndarray) -> np.ndarray:
        """Update raw material inventory and handle production constraints"""
        rm_needed = np.sum(planned_production)

        if self.rm_inventory >= rm_needed:
            self.rm_inventory -= rm_needed
            actual_production = planned_production
        else:
            scale_factor = self.rm_inventory / rm_needed if rm_needed > 0 else 0
            actual_production = planned_production * scale_factor
            self.rm_inventory = 0
            self.rm_stockout_count += 1

        return actual_production

    def _update_rm_pipeline(self, new_order: float):
        """Update raw material pipeline with FIFO logic"""
        for d in range(self.rm_lead_time - 1):
            self.rm_pipeline[d] = self.rm_pipeline[d + 1]
        self.rm_pipeline[-1] = new_order
        rm_arrival = self.rm_pipeline[0]
        self.rm_inventory += rm_arrival
        self.rm_pipeline[0] = 0

    def _get_obs(self) -> Dict[str, np.ndarray]:
        """Compile current system state into observations for both agents"""
        return {
            'fg_agent': self._get_fg_observation(),
            'rm_agent': self._get_rm_observation()
        }

    def _generate_demand_forecast(self) -> np.ndarray:
        """Generate demand forecast using simple moving average"""
        if not self.demand_history:
            return np.zeros((self.num_products, self.cfg.FORECAST_HORIZON))

        history_array = np.array(self.demand_history)
        forecast = []
        for product_idx in range(self.num_products):
            product_history = history_array[:, product_idx]
            product_forecast = np.full(self.cfg.FORECAST_HORIZON, product_history[-1])
            forecast.append(product_forecast)

        return np.array(forecast, dtype=np.float32)

    def render(self, mode='human'):
        """Render the current state of the environment"""
        print(f"\n== Step {self.current_step} ==")
        print(f"FG Inventory: {self.fg_inventory}")
        print(f"RM Inventory: {self.rm_inventory:.1f}")
        print(f"FG Pipeline shape: {self.fg_pipeline.shape}")
        print(f"RM Pipeline: {self.rm_pipeline}")

    def close(self):
        """Clean up environment resources"""
        pass

In [ ]:
%%writefile utils/visualization.py
"""
Visualization utilities for the inventory management environment
"""

import matplotlib.pyplot as plt
import numpy as np
from typing import List, Dict

def plot_environment_run(history: List[Dict], title: str = "Inventory Management Simulation"):
    """
    Plot the results of an environment run

    Args:
        history: List of info dictionaries from each step
        title: Plot title
    """
    steps = len(history)
    if steps == 0:
        return

    # Extract data
    fg_inventory_history = np.array([step['fg_inventory'] for step in history])
    rm_inventory_history = np.array([step['rm_inventory'] for step in history])
    lost_sales_history = np.array([step['lost_sales'] for step in history])
    total_costs = np.array([step['total_cost'] for step in history])

    num_products = fg_inventory_history.shape[1]

    # Create subplots
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    fig.suptitle(title, fontsize=16)

    # Plot 1: FG Inventory levels
    for product in range(num_products):
        axes[0, 0].plot(fg_inventory_history[:, product], label=f'Product {product+1}')
    axes[0, 0].set_title('Finished Goods Inventory')
    axes[0, 0].set_xlabel('Step')
    axes[0, 0].set_ylabel('Inventory Level')
    axes[0, 0].legend()
    axes[0, 0].grid(True)

    # Plot 2: RM Inventory
    axes[0, 1].plot(rm_inventory_history)
    axes[0, 1].set_title('Raw Material Inventory')
    axes[0, 1].set_xlabel('Step')
    axes[0, 1].set_ylabel('Inventory Level')
    axes[0, 1].grid(True)

    # Plot 3: Lost Sales
    for product in range(num_products):
        axes[0, 2].plot(lost_sales_history[:, product], label=f'Product {product+1}')
    axes[0, 2].set_title('Lost Sales')
    axes[0, 2].set_xlabel('Step')
    axes[0, 2].set_ylabel('Lost Sales Quantity')
    axes[0, 2].legend()
    axes[0, 2].grid(True)

    # Plot 4: Total Cost
    axes[1, 0].plot(total_costs)
    axes[1, 0].set_title('Total System Cost')
    axes[1, 0].set_xlabel('Step')
    axes[1, 0].set_ylabel('Cost')
    axes[1, 0].grid(True)

    # Plot 5: Cumulative Cost
    cumulative_cost = np.cumsum(total_costs)
    axes[1, 1].plot(cumulative_cost)
    axes[1, 1].set_title('Cumulative System Cost')
    axes[1, 1].set_xlabel('Step')
    axes[1, 1].set_ylabel('Cumulative Cost')
    axes[1, 1].grid(True)

    # Print summary statistics
    print(f"\n=== Simulation Summary ({steps} steps) ===")
    print(f"Total Cost: {np.sum(total_costs):.2f}")
    print(f"Average Cost per Step: {np.mean(total_costs):.2f}")
    print(f"Total Lost Sales: {np.sum(lost_sales_history):.2f}")
    print(f"Average FG Inventory: {np.mean(fg_inventory_history):.2f}")
    print(f"Average RM Inventory: {np.mean(rm_inventory_history):.2f}")

# Test the visualization with a short simulation
if __name__ == "__main__":
    from environment.inventory_env import InventoryMAMDPEnv

    env = InventoryMAMDPEnv()
    obs = env.reset()

    history = []
    for step in range(100):  # Short test run
        # Simple heuristic: order based on current inventory
        fg_action = np.maximum(0, 10 - env.fg_inventory)
        rm_action = np.array([max(0, 20 - env.rm_inventory)])

        joint_action = {'fg_agent': fg_action, 'rm_agent': rm_action}
        obs, reward, done, info = env.step(joint_action)
        history.append(info)

        if done:
            break

    plot_environment_run(history, "Basic Heuristic Policy Test")

In [ ]:
%%writefile models/networks.py
"""
neural network architectures with proper error handling
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from typing import Tuple, Optional
import configs.config as config

class TimeSeriesEncoder(nn.Module):
    """WaveNet-style dilated causal convolutions for time-series feature extraction"""

    def __init__(self, input_dim: int, config_obj=None):
        super(TimeSeriesEncoder, self).__init__()

        self.cfg = config_obj if config_obj else config.config.model

        self.input_dim = input_dim
        self.hidden_dim = self.cfg.HIDDEN_DIM
        self.out_channels = self.cfg.CNN_OUT_CHANNELS
        self.kernel_size = self.cfg.CNN_KERNEL_SIZE
        self.num_layers = self.cfg.CNN_NUM_LAYERS

        # Input projection - FIXED: handle variable input dimensions
        self.input_conv = nn.Conv1d(
            in_channels=input_dim,  # This should match the feature dimension
            out_channels=self.out_channels,
            kernel_size=1
        )

        # Dilated causal convolution layers
        self.dilated_convs = nn.ModuleList()
        self.residual_convs = nn.ModuleList()
        self.skip_convs = nn.ModuleList()

        for i in range(self.num_layers):
            dilation = 2 ** i

            # Dilated causal convolution
            dilated_conv = nn.Conv1d(
                in_channels=self.out_channels,
                out_channels=self.out_channels,
                kernel_size=self.kernel_size,
                dilation=dilation,
                padding=0
            )
            self.dilated_convs.append(dilated_conv)

            # Residual connection
            residual_conv = nn.Conv1d(
                in_channels=self.out_channels,
                out_channels=self.out_channels,
                kernel_size=1
            )
            self.residual_convs.append(residual_conv)

            # Skip connection
            skip_conv = nn.Conv1d(
                in_channels=self.out_channels,
                out_channels=self.out_channels,
                kernel_size=1
            )
            self.skip_convs.append(skip_conv)

        # Output projection
        self.output_conv = nn.Conv1d(
            in_channels=self.out_channels,
            out_channels=self.hidden_dim,
            kernel_size=1
        )

        # Activation functions
        self.activation = nn.ReLU()
        self.tanh = nn.Tanh()

    def _causal_pad(self, x: torch.Tensor, dilation: int) -> torch.Tensor:
        """Apply causal padding to the left of the input sequence"""
        padding = (self.kernel_size - 1) * dilation
        return F.pad(x, (padding, 0))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass through the time-series encoder"""

        if x.numel() == 0:
            # Return zeros if no forecast data
            batch_size = x.shape[0] if len(x.shape) > 0 else 1
            return torch.zeros(batch_size, self.hidden_dim, device=x.device)

        # Input shape: (batch_size, seq_len, input_dim)
        # Transpose to: (batch_size, input_dim, seq_len) for Conv1d
        x = x.transpose(1, 2)

        # Input projection
        x = self.input_conv(x)
        residual = x

        skip_connections = []

        # Dilated causal convolution layers
        for i, (dilated_conv, residual_conv, skip_conv) in enumerate(zip(
            self.dilated_convs, self.residual_convs, self.skip_convs)):

            dilation = 2 ** i

            # Apply causal padding
            x_padded = self._causal_pad(x, dilation)

            # Dilated convolution
            x_out = dilated_conv(x_padded)
            x_out = self.tanh(x_out)

            # Residual connection
            residual_out = residual_conv(x_out)

            # Skip connection
            skip_out = skip_conv(x_out)
            skip_connections.append(skip_out)

            # Update for next layer
            x = x + residual_out
            residual = x_out

        # Combine skip connections
        if skip_connections:
            x = torch.stack(skip_connections).sum(dim=0)

        # Output projection
        x = self.output_conv(x)
        x = self.activation(x)

        # Global average pooling over time dimension
        x = F.adaptive_avg_pool1d(x, 1).squeeze(-1)

        return x

class FGActorCritic(nn.Module):
    """FG Actor-Critic network with simplified processing for new state structure"""

    def __init__(self, state_dim: int, action_dim: int, config_obj=None, output_multiplier=0.3):
        super(FGActorCritic, self).__init__()

        self.cfg = config_obj if config_obj else config.config.model

        self.state_dim = state_dim
        self.action_dim = action_dim

        # Simple MLP architecture - no time-series encoder for now
        # The enhanced state already has aggregated temporal features
        self.shared_mlp = nn.Sequential(
            nn.Linear(state_dim, self.cfg.HIDDEN_DIM),
            nn.ReLU(),
            nn.Linear(self.cfg.HIDDEN_DIM, self.cfg.HIDDEN_DIM),
            nn.ReLU(),
            nn.Linear(self.cfg.HIDDEN_DIM, self.cfg.HIDDEN_DIM),
            nn.ReLU()
        )

        # Actor head - outputs mean and log_std for each action dimension
        self.actor_mean = nn.Linear(self.cfg.HIDDEN_DIM, action_dim)
        self.actor_log_std = nn.Linear(self.cfg.HIDDEN_DIM, action_dim)

        # Critic head - outputs state value
        self.critic = nn.Linear(self.cfg.HIDDEN_DIM, 1)

        self.output_multiplier = output_multiplier

        # Initialize weights
        self._initialize_weights()

    def _initialize_weights(self):
        """Initialize network weights"""
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.orthogonal_(module.weight, gain=0.1)
                nn.init.constant_(module.bias, 0.0)

    def forward(self, state: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """Forward pass through the FG agent network"""

        # Direct MLP processing - the state already contains enhanced features
        shared_features = self.shared_mlp(state)

        # Actor outputs
        mu = self.actor_mean(shared_features) * self.output_multiplier
        log_std = self.actor_log_std(shared_features)

        # Ensure positive standard deviation using softplus
        sigma = F.softplus(log_std) + 1e-4

        # Critic output
        value = self.critic(shared_features)

        return mu, sigma, value

    def get_action(self, state: torch.Tensor, deterministic: bool = False) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """Sample an action from the policy distribution"""
        mu, sigma, value = self.forward(state)

        if deterministic:
            action = mu
            log_prob = torch.zeros_like(mu)
        else:
            # Create normal distribution
            dist = torch.distributions.Normal(mu, sigma)
            # Sample action
            action = dist.rsample()
            # Calculate log probability
            log_prob = dist.log_prob(action).sum(dim=-1, keepdim=True)
            # Ensure actions are non-negative
            action = F.softplus(action)

        return action, log_prob, value

class RMActorCritic(nn.Module):
    """RM Actor-Critic network - simple MLP"""

    def __init__(self, state_dim: int, action_dim: int, config_obj=None):
        super(RMActorCritic, self).__init__()

        self.cfg = config_obj if config_obj else config.config.model

        self.state_dim = state_dim
        self.action_dim = action_dim

        # Simple MLP architecture
        self.shared_mlp = nn.Sequential(
            nn.Linear(state_dim, self.cfg.HIDDEN_DIM),
            nn.ReLU(),
            nn.Linear(self.cfg.HIDDEN_DIM, self.cfg.HIDDEN_DIM),
            nn.ReLU(),
            nn.Linear(self.cfg.HIDDEN_DIM, self.cfg.HIDDEN_DIM),
            nn.ReLU()
        )

        # Actor head
        self.actor_mean = nn.Linear(self.cfg.HIDDEN_DIM, action_dim)
        self.actor_log_std = nn.Linear(self.cfg.HIDDEN_DIM, action_dim)

        # Critic head
        self.critic = nn.Linear(self.cfg.HIDDEN_DIM, 1)

        # Initialize weights
        self._initialize_weights()

    def _initialize_weights(self):
        """Initialize network weights"""
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.orthogonal_(module.weight, gain=0.01)
                nn.init.constant_(module.bias, 0.0)

    def forward(self, state: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """Forward pass through the RM agent network"""
        # Shared MLP processing
        shared_features = self.shared_mlp(state)

        # Actor outputs
        mu = self.actor_mean(shared_features)
        log_std = self.actor_log_std(shared_features)

        # Ensure positive standard deviation
        sigma = F.softplus(log_std) + 1e-4

        # Critic output
        value = self.critic(shared_features)

        return mu, sigma, value

    def get_action(self, state: torch.Tensor, deterministic: bool = False) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """Sample an action from the policy distribution"""
        mu, sigma, value = self.forward(state)

        if deterministic:
            action = mu
            log_prob = torch.zeros_like(mu)
        else:
            dist = torch.distributions.Normal(mu, sigma)
            action = dist.rsample()
            log_prob = dist.log_prob(action).sum(dim=-1, keepdim=True)
            # Ensure actions are non-negative
            action = F.softplus(action)

        return action, log_prob, value

In [ ]:
%%writefile agents/ppo_agent.py
"""
PPO agent with IMPROVED monotonicity regularization
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
from typing import List, Dict, Tuple, Optional
import configs.config as config

class PPOMemory:
    """Experience replay buffer"""
    def __init__(self):
        self.states = []
        self.actions = []
        self.log_probs = []
        self.rewards = []
        self.values = []
        self.dones = []
        self.ptr = 0

    def store(self, state: np.ndarray, action: np.ndarray, log_prob: np.ndarray,
              reward: dict, value: float, done: bool):
        self.states.append(np.asarray(state, dtype=np.float32))
        self.actions.append(np.asarray(action, dtype=np.float32).flatten())
        self.log_probs.append(np.asarray(log_prob, dtype=np.float32).flatten())
        self.rewards.append(reward)
        self.values.append(float(value))
        self.dones.append(bool(done))
        self.ptr += 1

    def get_batches(self, batch_size: int) -> Tuple[np.ndarray, ...]:
        if len(self.states) == 0:
            return [], [], [], [], [], []

        states = np.array(self.states)
        max_action_dim = max(action.shape[0] for action in self.actions) if self.actions else 1
        max_log_prob_dim = max(log_prob.shape[0] for log_prob in self.log_probs) if self.log_probs else 1

        actions_array = []
        for action in self.actions:
            padded_action = np.zeros(max_action_dim, dtype=np.float32)
            padded_action[:action.shape[0]] = action
            actions_array.append(padded_action)

        log_probs_array = []
        for log_prob in self.log_probs:
            padded_log_prob = np.zeros(max_log_prob_dim, dtype=np.float32)
            padded_log_prob[:log_prob.shape[0]] = log_prob
            log_probs_array.append(padded_log_prob)

        actions = np.array(actions_array)
        log_probs_old = np.array(log_probs_array)
        rewards = np.array(self.rewards)
        values = np.array(self.values)
        dones = np.array(self.dones)

        advantages, returns = self._compute_advantages(rewards, values, dones)

        if len(advantages) > 1:
            advantages = (advantages - np.mean(advantages)) / (np.std(advantages) + 1e-8)

        batch_start = np.arange(0, len(states), batch_size)
        indices = np.arange(len(states), dtype=np.int64)
        np.random.shuffle(indices)
        batches = [indices[i:i+batch_size] for i in batch_start]

        return states, actions, log_probs_old, advantages, returns, batches

    def _compute_advantages(self, rewards: np.ndarray, values: np.ndarray,
                          dones: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        cfg = config.config.ppo
        advantages = np.zeros_like(rewards)
        returns = np.zeros_like(rewards)
        last_advantage = 0
        next_value = 0

        for t in reversed(range(len(rewards))):
            if t == len(rewards) - 1:
                next_non_terminal = 1.0 - dones[t]
                next_values = next_value
            else:
                next_non_terminal = 1.0 - dones[t]
                next_values = values[t + 1]

            delta = rewards[t] + cfg.GAMMA * next_values * next_non_terminal - values[t]
            advantages[t] = delta + cfg.GAMMA * cfg.LAMBDA_GAE * next_non_terminal * last_advantage
            last_advantage = advantages[t]
            returns[t] = advantages[t] + values[t]

        return advantages, returns

    def clear(self):
        self.states = []
        self.actions = []
        self.log_probs = []
        self.rewards = []
        self.values = []
        self.dones = []
        self.ptr = 0

    def __len__(self):
        return len(self.states)

class PPOAgent:
    """
    IMPROVED VERSION: PPO agent with robust monotonicity regularization
    """

    def __init__(self, actor_critic_network: nn.Module, agent_type: str = "fg_agent", config_obj=None):
        self.cfg = config_obj if config_obj else config.config
        self.ppo_cfg = self.cfg.ppo
        self.model_cfg = self.cfg.model
        self.agent_type = agent_type

        self.device = torch.device("cuda" if torch.cuda.is_available() and self.cfg.colab.USE_GPU else "cpu")

        # Network
        self.actor_critic = actor_critic_network.to(self.device)

        # Optimizers
        self.actor_optimizer = optim.Adam(
            list(self.actor_critic.actor_mean.parameters()) +
            list(self.actor_critic.actor_log_std.parameters()),
            lr=self.model_cfg.LEARNING_RATE_ACTOR,
            eps=self.model_cfg.OPTIMIZER_EPS
        )
        self.critic_optimizer = optim.Adam(
            self.actor_critic.critic.parameters(),
            lr=self.model_cfg.LEARNING_RATE_CRITIC,
            eps=self.model_cfg.OPTIMIZER_EPS
        )

        # Memory buffer
        self.memory = PPOMemory()

        # Training stats
        self.training_steps = 0

        if self.cfg.colab.DEBUG_MODE:
            print(f"PPOAgent ({agent_type}) initialized on {self.device}")

    def _compute_monotonicity_penalty(self, states: torch.Tensor, actions_mu: torch.Tensor) -> torch.Tensor:
        """
        IMPROVED: Robust monotonicity regularization
        Only applied to FG agent, penalizes d_action/d_inventory > 0
        """
        if self.agent_type != "fg_agent":
            # Only apply monotonicity penalty to FG agent
            return torch.tensor(0.0, device=self.device)

        try:
            num_products = config.config.env.NUM_PRODUCTS

            # For FG agent, inventory positions are the first num_products elements
            inventory_indices = list(range(num_products))

            monotonicity_penalties = []

            for product_idx, inv_idx in enumerate(inventory_indices):
                if inv_idx >= states.shape[1]:
                    continue  # Skip if inventory index is out of bounds

                # Compute gradient: d(action[product_idx]) / d(inventory[inv_idx])
                if states.requires_grad:
                    # Reset gradients
                    if states.grad is not None:
                        states.grad.zero_()

                # Ensure we're tracking gradients
                states_requires_grad = states.requires_grad
                if not states_requires_grad:
                    states.requires_grad_(True)

                # Forward pass to recompute actions with gradient tracking
                mu_recompute, _, _ = self.actor_critic(states)

                # Compute gradient for this specific product
                grad_outputs = torch.ones_like(mu_recompute[:, product_idx])

                gradients = torch.autograd.grad(
                    outputs=mu_recompute[:, product_idx],
                    inputs=states,
                    grad_outputs=grad_outputs,
                    create_graph=True,
                    retain_graph=True,
                    only_inputs=True
                )

                if gradients[0] is not None:
                    # Extract gradient for this inventory position
                    inventory_grad = gradients[0][:, inv_idx]

                    # Penalize positive gradients (increasing order when inventory increases)
                    # Using smooth L2 penalty for positive gradients only
                    positive_grads = F.relu(inventory_grad)
                    product_penalty = (positive_grads ** 2).mean()
                    monotonicity_penalties.append(product_penalty)

                # Restore original requires_grad state
                if not states_requires_grad:
                    states.requires_grad_(False)

            if monotonicity_penalties:
                total_penalty = torch.stack(monotonicity_penalties).mean()
            else:
                total_penalty = torch.tensor(0.0, device=self.device)

            return total_penalty

        except Exception as e:
            print(f"▲ Monotonicity penalty computation failed: {e}")
            return torch.tensor(0.0, device=self.device)

    def store_transition(self, state: np.ndarray, action: np.ndarray,
                        log_prob: np.ndarray, reward: dict,
                        value: float, done: bool):
        state = np.asarray(state, dtype=np.float32)
        action = np.asarray(action, dtype=np.float32)
        log_prob = np.asarray(log_prob, dtype=np.float32)
        self.memory.store(state, action, log_prob, reward, value, done)

    def act(self, state: np.ndarray, deterministic: bool = False) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        state_tensor = torch.FloatTensor(state).unsqueeze(0).to(self.device)

        with torch.no_grad():
            action, log_prob, value = self.actor_critic.get_action(
                state_tensor, deterministic=deterministic
            )
            action_np = action.cpu().numpy()[0]
            log_prob_np = log_prob.cpu().numpy()[0]
            value_np = value.cpu().numpy()[0]

        return action_np, log_prob_np, value_np

    def update(self) -> Dict[str, float]:
        """Perform PPO update with robust monotonicity regularization"""
        if len(self.memory) < self.ppo_cfg.MINIBATCH_SIZE:
            return {"skipped": 1.0}

        try:
            # Get training data
            states, actions, log_probs_old, advantages, returns, batches = self.memory.get_batches(
                self.ppo_cfg.MINIBATCH_SIZE
            )
            if len(states) == 0:
                return {"skipped": 1.0}

            # Convert to tensors
            states_tensor = torch.FloatTensor(states).to(self.device)
            actions_tensor = torch.FloatTensor(actions).to(self.device)
            log_probs_old_tensor = torch.FloatTensor(log_probs_old).to(self.device)
            advantages_tensor = torch.FloatTensor(advantages).to(self.device)
            returns_tensor = torch.FloatTensor(returns).to(self.device)

            # Training statistics
            stats = {
                'policy_loss': 0.0,
                'value_loss': 0.0,
                'entropy_loss': 0.0,
                'total_loss': 0.0,
                'clip_fraction': 0.0,
                'monotonicity_penalty': 0.0
            }

            # PPO epochs
            update_count = 0
            for _ in range(self.ppo_cfg.PPO_EPOCHS):
                for batch_idx in batches:
                    if len(batch_idx) == 0:
                        continue

                    # Get batch data
                    batch_states = states_tensor[batch_idx].clone().detach().requires_grad_(True)
                    batch_actions = actions_tensor[batch_idx]
                    batch_log_probs_old = log_probs_old_tensor[batch_idx]
                    batch_advantages = advantages_tensor[batch_idx]
                    batch_returns = returns_tensor[batch_idx]

                    # Get current policy outputs
                    mu, sigma, values = self.actor_critic(batch_states)

                    # Handle different action dimensions
                    action_dim = min(mu.shape[1], batch_actions.shape[1])
                    mu = mu[:, :action_dim]
                    sigma = sigma[:, :action_dim]
                    batch_actions = batch_actions[:, :action_dim]

                    if batch_log_probs_old.dim() > 1:
                        batch_log_probs_old = batch_log_probs_old[:, :action_dim]

                    # Create distribution and calculate new log probs
                    dist = torch.distributions.Normal(mu, sigma)
                    log_probs_new = dist.log_prob(batch_actions).sum(dim=1, keepdim=True)

                    # Calculate probability ratio
                    ratio = torch.exp(log_probs_new - batch_log_probs_old)

                    # PPO Clipped Objective
                    surr1 = ratio * batch_advantages.unsqueeze(1)
                    surr2 = torch.clamp(ratio, 1 - self.ppo_cfg.EPSILON_CLIP,
                                      1 + self.ppo_cfg.EPSILON_CLIP) * batch_advantages.unsqueeze(1)
                    policy_loss = -torch.min(surr1, surr2).mean()

                    # Value function loss
                    value_loss = F.mse_loss(values, batch_returns.unsqueeze(1))

                    # Entropy bonus
                    entropy_loss = -dist.entropy().mean()

                    # Monotonicity regularization (only for FG agent)
                    monotonicity_penalty = self._compute_monotonicity_penalty(batch_states, mu)

                    # Total loss with monotonicity regularization
                    total_loss = (policy_loss +
                                self.ppo_cfg.VALUE_COEFF * value_loss +
                                self.ppo_cfg.ENTROPY_COEFF * entropy_loss +
                                self.cfg.training.LAMBDA_MONO * monotonicity_penalty)

                    # Gradient updates
                    self.actor_optimizer.zero_grad()
                    self.critic_optimizer.zero_grad()

                    total_loss.backward()

                    # Gradient clipping
                    torch.nn.utils.clip_grad_norm_(
                        list(self.actor_critic.actor_mean.parameters()) +
                        list(self.actor_critic.actor_log_std.parameters()),
                        self.ppo_cfg.MAX_GRAD_NORM
                    )

                    torch.nn.utils.clip_grad_norm_(
                        self.actor_critic.critic.parameters(),
                        self.ppo_cfg.MAX_GRAD_NORM
                    )

                    self.actor_optimizer.step()
                    self.critic_optimizer.step()

                    # Update statistics
                    stats['policy_loss'] += policy_loss.item()
                    stats['value_loss'] += value_loss.item()
                    stats['entropy_loss'] += entropy_loss.item()
                    stats['total_loss'] += total_loss.item()
                    stats['monotonicity_penalty'] += monotonicity_penalty.item()

                    # Calculate clip fraction
                    clipped = torch.logical_or(
                        ratio < (1 - self.ppo_cfg.EPSILON_CLIP),
                        ratio > (1 + self.ppo_cfg.EPSILON_CLIP)
                    )
                    stats['clip_fraction'] += clipped.float().mean().item()

                    self.training_steps += 1
                    update_count += 1

            # Average statistics
            if update_count > 0:
                for key in stats:
                    if key != 'skipped':
                        stats[key] /= update_count

            # Clear memory
            self.memory.clear()

            return stats

        except Exception as e:
            print(f"▲ PPO update failed: {e}")
            self.memory.clear()
            return {'error': 1.0}

    def save_checkpoint(self, filepath: str):
        checkpoint = {
            'actor_critic_state_dict': self.actor_critic.state_dict(),
            'actor_optimizer_state_dict': self.actor_optimizer.state_dict(),
            'critic_optimizer_state_dict': self.critic_optimizer.state_dict(),
            'training_steps': self.training_steps
        }
        torch.save(checkpoint, filepath)

    def load_checkpoint(self, filepath: str):
        checkpoint = torch.load(filepath, map_location=self.device)
        self.actor_critic.load_state_dict(checkpoint['actor_critic_state_dict'])
        self.actor_optimizer.load_state_dict(checkpoint['actor_optimizer_state_dict'])
        self.critic_optimizer.load_state_dict(checkpoint['critic_optimizer_state_dict'])
        self.training_steps = checkpoint['training_steps']

In [ ]:
%%writefile utils/training_utils.py
"""
Utility functions for multi-agent training coordination
"""

import numpy as np
import torch
from typing import Dict, List, Tuple
from agents.ppo_agent import PPOAgent

class MultiAgentTrainer:
    """
    Coordinates training of multiple PPO agents
    """

    def __init__(self, agents: Dict[str, PPOAgent], agent_names: List[str]):
        self.agents = agents
        self.agent_names = agent_names

    def act(self, observations: Dict[str, np.ndarray], deterministic: bool = False) -> Dict[str, np.ndarray]:
        """Get actions from all agents"""
        actions = {}
        for name in self.agent_names:
            action, _, _ = self.agents[name].act(observations[name], deterministic)
            actions[name] = action
        return actions

    def store_transitions(self, observations: Dict[str, np.ndarray],
                         actions: Dict[str, np.ndarray],
                         log_probs: Dict[str, np.ndarray],
                         reward: float, values: Dict[str, np.ndarray], done: bool):
        """Store transitions for all agents"""
        for name in self.agent_names:
            self.agents[name].store_transition(
                observations[name], actions[name], log_probs[name], reward, values[name], done
            )

    def update_all(self) -> Dict[str, Dict[str, float]]:
        """Update all agents and return combined statistics"""
        stats = {}
        for name in self.agent_names:
            stats[name] = self.agents[name].update()
        return stats

    def get_values(self, observations: Dict[str, np.ndarray]) -> Dict[str, np.ndarray]:
        """Get value estimates from all agents"""
        values = {}
        for name in self.agent_names:
            _, _, value = self.agents[name].act(observations[name], deterministic=True)
            values[name] = value
        return values

    def save_checkpoints(self, base_path: str):
        """Save checkpoints for all agents"""
        for name in self.agent_names:
            filepath = f"{base_path}_{name}.pth"
            self.agents[name].save_checkpoint(filepath)

    def load_checkpoints(self, base_path: str):
        """Load checkpoints for all agents"""
        for name in self.agent_names:
            filepath = f"{base_path}_{name}.pth"
            self.agents[name].load_checkpoint(filepath)

def create_agents(env) -> Tuple[Dict[str, PPOAgent], List[str]]:
    """Create PPO agents for the inventory management environment"""
    from models.networks import FGActorCritic, RMActorCritic

    # Create networks
    fg_network = FGActorCritic(
        state_dim=env.observation_space['fg_agent'].shape[0],
        action_dim=env.action_space_fg.shape[0]
    )

    rm_network = RMActorCritic(
        state_dim=env.observation_space['rm_agent'].shape[0],
        action_dim=env.action_space_rm.shape[0]
    )

    # Create agents
    agents = {
        'fg_agent': PPOAgent(fg_network),
        'rm_agent': PPOAgent(rm_network)
    }

    agent_names = ['fg_agent', 'rm_agent']

    return agents, agent_names

print("Multi-agent training utilities created successfully!")

In [ ]:
%%writefile utils/heuristic_policy.py
"""
Simplified heuristic policy for effective imitation learning
"""

import numpy as np
from typing import Dict
import configs.config as config

def calculate_base_stock_action(state: np.ndarray, agent_type: str, config_obj=None) -> np.ndarray:
    """
    SIMPLIFIED VERSION: Base-stock policy that is easier to imitate
    """
    cfg = config_obj if config_obj else config.config

    if agent_type == 'fg_agent':
        return _fg_base_stock(state, cfg)
    elif agent_type == 'rm_agent':
        return _rm_base_stock(state, cfg)
    else:
        raise ValueError(f"Unknown agent type: {agent_type}")

def _fg_base_stock(state: np.ndarray, cfg) -> np.ndarray:
    """Enhanced base-stock policy for Finished Goods agent with new state structure"""

    num_products = cfg.env.NUM_PRODUCTS
    fg_lead_time = cfg.env.FG_LEAD_TIME

    # Extract from enhanced state structure (37 dimensions)
    # Structure: 5(inv) + 15(pipe) + 4(demand) + 3(hist) + 4(prod) + 2(temp) + 4(RM)
    inventory_start = 0
    inventory_end = num_products  # 5
    pipeline_end = inventory_end + num_products * fg_lead_time  # 5 + 15 = 20
    demand_metrics_start = pipeline_end  # 20
    historical_start = demand_metrics_start + 4  # 24
    production_start = historical_start + 3  # 27
    temporal_start = production_start + 4  # 31
    rm_info_start = temporal_start + 2  # 33

    current_inventory = state[inventory_start:inventory_end]
    pipeline = state[inventory_end:pipeline_end].reshape(num_products, fg_lead_time)

    # Extract key metrics from new state structure
    immediate_demand = state[demand_metrics_start + 3]  # Last element of demand metrics
    demand_variability = state[demand_metrics_start + 2]  # Std of demand
    rm_inventory = state[rm_info_start]  # First element of RM info
    rm_pipeline_total = state[rm_info_start + 1]  # Second element of RM info

    # Dynamic base-stock based on demand and RM availability
    safety_factor = 1.0 + (demand_variability / 10.0)  # Adjust based on demand variability
    base_stock_levels = np.array([max(15, immediate_demand * 2 * safety_factor)] * num_products)

    # Adjust based on RM availability - more sophisticated
    total_rm_available = rm_inventory + rm_pipeline_total
    total_needed = np.sum(base_stock_levels)
    rm_scale = min(1.0, total_rm_available / (total_needed + 1e-6))
    base_stock_levels = base_stock_levels * rm_scale

    # Order up to base-stock level
    inventory_position = current_inventory + np.sum(pipeline, axis=1)
    order_quantities = np.maximum(0, base_stock_levels - inventory_position)

    # Reasonable caps based on RM availability
    max_order_per_product = min(25.0, total_rm_available / num_products)
    order_quantities = np.minimum(order_quantities, max_order_per_product)

    # Small noise for diversity
    noise = np.random.normal(0, 0.3, size=order_quantities.shape)
    order_quantities = np.maximum(0, order_quantities + noise)

    return order_quantities.astype(np.float32)

def _rm_base_stock(state: np.ndarray, cfg) -> np.ndarray:
    """Simple, smooth base-stock policy for Raw Material agent"""
    rm_lead_time = cfg.env.RM_LEAD_TIME
    forecast_horizon = cfg.env.FORECAST_HORIZON
    num_products = cfg.env.NUM_PRODUCTS

    # Extract state components
    current_inventory = state[0]
    pipeline = state[1:1 + rm_lead_time]
    production_signal = state[1 + rm_lead_time]
    aggregated_forecast = state[1 + rm_lead_time + 1:1 + rm_lead_time + 1 + forecast_horizon]

    # Calculate inventory position
    inventory_position = current_inventory + np.sum(pipeline)

    # Simple target calculation
    if len(aggregated_forecast) > 0:
        avg_daily_demand = np.mean(aggregated_forecast)
    else:
        avg_daily_demand = 8.0

    # Simple target: cover lead time demand + safety stock
    lead_time_demand = avg_daily_demand * num_products * rm_lead_time
    safety_stock = avg_daily_demand * num_products * 3  # 3 days of safety stock

    target_inventory = lead_time_demand + safety_stock

    # Order up to target
    order_quantity = max(0, target_inventory - inventory_position)

    # Small noise for diversity
    noise = np.random.normal(0, 1.0)
    order_quantity = max(0, order_quantity + noise)

    return np.array([order_quantity], dtype=np.float32)

def get_heuristic_action(state_dict: Dict[str, np.ndarray], config_obj=None) -> Dict[str, np.ndarray]:
    """Simplified heuristic policy for both agents"""
    cfg = config_obj if config_obj else config.config

    fg_action = _fg_base_stock(state_dict['fg_agent'], cfg)
    rm_action = _rm_base_stock(state_dict['rm_agent'], cfg)

    return {
        'fg_agent': fg_action,
        'rm_agent': rm_action
    }

In [ ]:
%%writefile utils/evaluation.py
"""
Comprehensive evaluation and visualization utilities for multi-agent inventory management
"""

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from typing import List, Dict, Tuple, Optional
import seaborn as sns
from collections import defaultdict
import torch

from utils.visualization import plot_environment_run

def plot_training_progress(training_history: List[Dict],
                         window_size: int = 10,
                         title: str = "Training Progress Analysis"):
    """
    Comprehensive training progress visualization

    Args:
        training_history: List of episode statistics from training
        window_size: Window for moving averages
        title: Plot title
    """
    if not training_history:
        print("No training data available")
        return

    # Convert to DataFrame for easier analysis
    df = pd.DataFrame(training_history)

    print(df.head())

    # Create subplots - increased to 3 rows to accommodate new plots
    fig, axes = plt.subplots(3, 3, figsize=(18, 15))
    fig.suptitle(title, fontsize=16, fontweight='bold')

    # Plot 1: Rewards over time
    axes[0, 0].plot(df.index, df.get('total_reward', []), 'b-', alpha=0.3, label='Raw')
    if len(df) > window_size:
        rewards_ma = df['total_reward'].rolling(window=window_size).mean()
        axes[0, 0].plot(df.index[window_size-1:], rewards_ma[window_size-1:],
                       'b-', linewidth=2, label=f'MA({window_size})')
    axes[0, 0].set_title('Episode Rewards')
    axes[0, 0].set_xlabel('Episode')
    axes[0, 0].set_ylabel('Total Reward')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

    # Plot 2: Costs over time
    axes[0, 1].plot(df.index, df.get('total_cost', []), 'r-', alpha=0.3, label='Raw')
    if len(df) > window_size:
        costs_ma = df['total_cost'].rolling(window=window_size).mean()
        axes[0, 1].plot(df.index[window_size-1:], costs_ma[window_size-1:],
                       'r-', linewidth=2, label=f'MA({window_size})')
    axes[0, 1].set_title('Episode Costs')
    axes[0, 1].set_xlabel('Episode')
    axes[0, 1].set_ylabel('Total Cost')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)

    # Plot 3: Lost Sales over time
    axes[0, 2].plot(df.index, df.get('total_lost_sales', []), 'orange', alpha=0.3, label='Raw')
    if len(df) > window_size:
        lost_sales_ma = df['total_lost_sales'].rolling(window=window_size).mean()
        axes[0, 2].plot(df.index[window_size-1:], lost_sales_ma[window_size-1:],
                       'orange', linewidth=2, label=f'MA({window_size})')
    axes[0, 2].set_title('Lost Sales per Episode')
    axes[0, 2].set_xlabel('Episode')
    axes[0, 2].set_ylabel('Lost Sales Quantity')
    axes[0, 2].legend()
    axes[0, 2].grid(True, alpha=0.3)

    # Plot 4: RM Stockouts over time
    axes[1, 0].plot(df.index, df.get('total_rm_stockouts', []), 'purple', alpha=0.3, label='Raw')
    if len(df) > window_size:
        stockouts_ma = df['total_rm_stockouts'].rolling(window=window_size).mean()
        axes[1, 0].plot(df.index[window_size-1:], stockouts_ma[window_size-1:],
                       'purple', linewidth=2, label=f'MA({window_size})')
    axes[1, 0].set_title('RM Stockouts per Episode')
    axes[1, 0].set_xlabel('Episode')
    axes[1, 0].set_ylabel('RM Stockouts')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)

    # Plot 5: Episode lengths
    axes[1, 1].plot(df.index, df.get('episode_length', []), 'green', alpha=0.3, label='Raw')
    if len(df) > window_size:
        length_ma = df['episode_length'].rolling(window=window_size).mean()
        axes[1, 1].plot(df.index[window_size-1:], length_ma[window_size-1:],
                       'green', linewidth=2, label=f'MA({window_size})')
    axes[1, 1].set_title('Episode Lengths')
    axes[1, 1].set_xlabel('Episode')
    axes[1, 1].set_ylabel('Steps per Episode')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)

    # Plot 6: FG Actions over time
    if 'mean_fg_action' in df.columns:
        axes[1, 2].plot(df.index, df['mean_fg_action'], 'teal', alpha=0.3, label='Raw')
        if len(df) > window_size:
            fg_actions_ma = df['mean_fg_action'].rolling(window=window_size).mean()
            axes[1, 2].plot(df.index[window_size-1:], fg_actions_ma[window_size-1:],
                           'teal', linewidth=2, label=f'MA({window_size})')
        axes[1, 2].set_title('Average FG Action per Episode')
        axes[1, 2].set_xlabel('Episode')
        axes[1, 2].set_ylabel('Average FG Action')
        axes[1, 2].legend()
        axes[1, 2].grid(True, alpha=0.3)
    else:
        axes[1, 2].text(0.5, 0.5, 'FG Action Data\nNot Available',
                       ha='center', va='center', transform=axes[1, 2].transAxes)
        axes[1, 2].set_title('Average FG Action per Episode')

    # Plot 7: RM Actions over time
    if 'mean_rm_action' in df.columns:
        axes[2, 0].plot(df.index, df['mean_rm_action'], 'brown', alpha=0.3, label='Raw')
        if len(df) > window_size:
            rm_actions_ma = df['mean_rm_action'].rolling(window=window_size).mean()
            axes[2, 0].plot(df.index[window_size-1:], rm_actions_ma[window_size-1:],
                           'brown', linewidth=2, label=f'MA({window_size})')
        axes[2, 0].set_title('Average RM Action per Episode')
        axes[2, 0].set_xlabel('Episode')
        axes[2, 0].set_ylabel('Average RM Action')
        axes[2, 0].legend()
        axes[2, 0].grid(True, alpha=0.3)
    else:
        axes[2, 0].text(0.5, 0.5, 'RM Action Data\nNot Available',
                       ha='center', va='center', transform=axes[2, 0].transAxes)
        axes[2, 0].set_title('Average RM Action per Episode')

    # Plot 8: FG Action Variability
    if 'std_fg_action' in df.columns:
        axes[2, 1].plot(df.index, df['std_fg_action'], 'darkblue', alpha=0.3, label='Raw')
        if len(df) > window_size:
            fg_std_ma = df['std_fg_action'].rolling(window=window_size).mean()
            axes[2, 1].plot(df.index[window_size-1:], fg_std_ma[window_size-1:],
                           'darkblue', linewidth=2, label=f'MA({window_size})')
        axes[2, 1].set_title('FG Action Variability (Std Dev)')
        axes[2, 1].set_xlabel('Episode')
        axes[2, 1].set_ylabel('FG Action Std Dev')
        axes[2, 1].legend()
        axes[2, 1].grid(True, alpha=0.3)
    else:
        axes[2, 1].text(0.5, 0.5, 'FG Action Std Dev\nNot Available',
                       ha='center', va='center', transform=axes[2, 1].transAxes)
        axes[2, 1].set_title('FG Action Variability')

    # Plot 9: Performance metrics correlation
    if len(df) > 10:
        # Include action metrics in correlation if available
        correlation_columns = ['total_reward', 'total_cost', 'total_lost_sales', 'total_rm_stockouts']
        if 'mean_fg_action' in df.columns:
            correlation_columns.append('mean_fg_action')
        if 'mean_rm_action' in df.columns:
            correlation_columns.append('mean_rm_action')

        correlation_data = df[correlation_columns].corr()
        im = axes[2, 2].imshow(correlation_data, cmap='coolwarm', vmin=-1, vmax=1)
        axes[2, 2].set_title('Metrics Correlation')
        axes[2, 2].set_xticks(range(len(correlation_data.columns)))
        axes[2, 2].set_yticks(range(len(correlation_data.columns)))
        axes[2, 2].set_xticklabels(correlation_data.columns, rotation=45)
        axes[2, 2].set_yticklabels(correlation_data.columns)

        # Add correlation values as text
        for i in range(len(correlation_data.columns)):
            for j in range(len(correlation_data.columns)):
                axes[2, 2].text(j, i, f'{correlation_data.iloc[i, j]:.2f}',
                               ha='center', va='center', fontweight='bold')

        plt.colorbar(im, ax=axes[2, 2])
    else:
        axes[2, 2].text(0.5, 0.5, 'Correlation Matrix\nNot Available\n(Need >10 episodes)',
                       ha='center', va='center', transform=axes[2, 2].transAxes)
        axes[2, 2].set_title('Metrics Correlation')

    plt.tight_layout()
    plt.show()

    # Print summary statistics
    print_summary_statistics(df)

def print_summary_statistics(df: pd.DataFrame):
    """Print comprehensive training statistics"""
    print("\n" + "="*60)
    print("TRAINING SUMMARY STATISTICS")
    print("="*60)

    if 'total_reward' in df.columns:
        print(f"Rewards:    Mean = {df['total_reward'].mean():8.2f} ± {df['total_reward'].std():6.2f}")
        print(f"            Best  = {df['total_reward'].max():8.2f} (episode {df['total_reward'].idxmax()})")
        print(f"            Worst = {df['total_reward'].min():8.2f} (episode {df['total_reward'].idxmin()})")

    if 'total_cost' in df.columns:
        print(f"Costs:      Mean = {df['total_cost'].mean():8.2f} ± {df['total_cost'].std():6.2f}")

    if 'total_lost_sales' in df.columns:
        print(f"Lost Sales: Mean = {df['total_lost_sales'].mean():8.2f} ± {df['total_lost_sales'].std():6.2f}")

    if 'total_rm_stockouts' in df.columns:
        print(f"RM Stockouts: Mean = {df['total_rm_stockouts'].mean():8.2f} ± {df['total_rm_stockouts'].std():6.2f}")

    if 'episode_length' in df.columns:
        print(f"Episode Length: Mean = {df['episode_length'].mean():6.1f} steps")

def plot_agent_behavior_comparison(heuristic_history: List[Dict],
                                 trained_history: List[Dict],
                                 titles: Tuple[str, str] = ("Heuristic Policy", "Trained Policy")):
    """
    Compare heuristic vs trained policy performance

    Args:
        heuristic_history: Simulation history with heuristic policy
        trained_history: Simulation history with trained policy
        titles: Titles for the two policies
    """
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle('Policy Comparison: Heuristic vs Trained', fontsize=16, fontweight='bold')

    # Extract data
    heuristic_costs = [step['total_cost'] for step in heuristic_history]
    trained_costs = [step['total_cost'] for step in trained_history]

    heuristic_lost_sales = [np.sum(step['lost_sales']) for step in heuristic_history]
    trained_lost_sales = [np.sum(step['lost_sales']) for step in trained_history]

    heuristic_rm_inv = [step['rm_inventory'] for step in heuristic_history]
    trained_rm_inv = [step['rm_inventory'] for step in trained_history]

    heuristic_fg_inv = [np.mean(step['fg_inventory']) for step in heuristic_history]
    trained_fg_inv = [np.mean(step['fg_inventory']) for step in trained_history]

    # Plot 1: Cost comparison
    time_steps = range(len(heuristic_costs))
    axes[0, 0].plot(time_steps, heuristic_costs, 'r-', label=titles[0], alpha=0.7)
    axes[0, 0].plot(time_steps, trained_costs, 'b-', label=titles[1], alpha=0.7)
    axes[0, 0].set_title('Daily System Cost Comparison')
    axes[0, 0].set_xlabel('Time Step')
    axes[0, 0].set_ylabel('Cost')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

    # Plot 2: Lost sales comparison
    axes[0, 1].plot(time_steps, heuristic_lost_sales, 'r-', label=titles[0], alpha=0.7)
    axes[0, 1].plot(time_steps, trained_lost_sales, 'b-', label=titles[1], alpha=0.7)
    axes[0, 1].set_title('Lost Sales Comparison')
    axes[0, 1].set_xlabel('Time Step')
    axes[0, 1].set_ylabel('Lost Sales Quantity')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)

    # Plot 3: RM inventory comparison
    axes[1, 0].plot(time_steps, heuristic_rm_inv, 'r-', label=titles[0], alpha=0.7)
    axes[1, 0].plot(time_steps, trained_rm_inv, 'b-', label=titles[1], alpha=0.7)
    axes[1, 0].set_title('RM Inventory Level Comparison')
    axes[1, 0].set_xlabel('Time Step')
    axes[1, 0].set_ylabel('RM Inventory')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)

    # Plot 4: FG inventory comparison
    axes[1, 1].plot(time_steps, heuristic_fg_inv, 'r-', label=titles[0], alpha=0.7)
    axes[1, 1].plot(time_steps, trained_fg_inv, 'b-', label=titles[1], alpha=0.7)
    axes[1, 1].set_title('Average FG Inventory Comparison')
    axes[1, 1].set_xlabel('Time Step')
    axes[1, 1].set_ylabel('Average FG Inventory')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Print comparison statistics
    print_comparison_statistics(heuristic_history, trained_history, titles)

def print_comparison_statistics(heuristic_history: List[Dict],
                              trained_history: List[Dict],
                              titles: Tuple[str, str]):
    """Print detailed comparison statistics"""
    print("\n" + "="*60)
    print("POLICY COMPARISON STATISTICS")
    print("="*60)

    heuristic_costs = [step['total_cost'] for step in heuristic_history]
    trained_costs = [step['total_cost'] for step in trained_history]

    heuristic_lost_sales = [np.sum(step['lost_sales']) for step in heuristic_history]
    trained_lost_sales = [np.sum(step['lost_sales']) for step in trained_history]

    heuristic_rm_stockouts = [step.get('rm_stockout_count', 0) for step in heuristic_history]
    trained_rm_stockouts = [step.get('rm_stockout_count', 0) for step in trained_history]

    print(f"{'Metric':<25} {titles[0]:<15} {titles[1]:<15} {'Improvement':<15}")
    print("-" * 70)

    # Total cost comparison
    heuristic_total_cost = np.sum(heuristic_costs)
    trained_total_cost = np.sum(trained_costs)
    cost_improvement = ((heuristic_total_cost - trained_total_cost) / heuristic_total_cost) * 100
    print(f"{'Total Cost:':<25} {heuristic_total_cost:<15.2f} {trained_total_cost:<15.2f} {cost_improvement:>+6.1f}%")

    # Average daily cost
    heuristic_avg_cost = np.mean(heuristic_costs)
    trained_avg_cost = np.mean(trained_costs)
    avg_cost_improvement = ((heuristic_avg_cost - trained_avg_cost) / heuristic_avg_cost) * 100
    print(f"{'Avg Daily Cost:':<25} {heuristic_avg_cost:<15.2f} {trained_avg_cost:<15.2f} {avg_cost_improvement:>+6.1f}%")

    # Total lost sales
    heuristic_total_lost = np.sum(heuristic_lost_sales)
    trained_total_lost = np.sum(trained_lost_sales)
    lost_improvement = ((heuristic_total_lost - trained_total_lost) / heuristic_total_lost) * 100
    print(f"{'Total Lost Sales:':<25} {heuristic_total_lost:<15.2f} {trained_total_lost:<15.2f} {lost_improvement:>+6.1f}%")

    # RM stockouts
    heuristic_total_rm_stockouts = np.sum(heuristic_rm_stockouts)
    trained_total_rm_stockouts = np.sum(trained_rm_stockouts)
    rm_stockout_improvement = ((heuristic_total_rm_stockouts - trained_total_rm_stockouts) /
                              (heuristic_total_rm_stockouts + 1e-6)) * 100
    print(f"{'RM Stockouts:':<25} {heuristic_total_rm_stockouts:<15.0f} {trained_total_rm_stockouts:<15.0f} {rm_stockout_improvement:>+6.1f}%")

def plot_action_distribution(heuristic_actions: Dict,
                           trained_actions: Dict,
                           agent_type: str = "fg_agent"):
    """
    Plot action distribution comparison between heuristic and trained policies

    Args:
        heuristic_actions: Dictionary of actions from heuristic policy
        trained_actions: Dictionary of actions from trained policy
        agent_type: Type of agent ('fg_agent' or 'rm_agent')
    """
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle(f'{agent_type.upper()} Action Distribution Comparison', fontsize=14)

    if agent_type == "fg_agent":
        # FG agent actions (multiple products)
        heuristic_fg = np.array(heuristic_actions['fg_agent'])
        trained_fg = np.array(trained_actions['fg_agent'])

        num_products = heuristic_fg.shape[1] if len(heuristic_fg.shape) > 1 else 1

        if num_products > 1:
            # Plot distribution for each product
            for product in range(num_products):
                axes[0].hist(heuristic_fg[:, product], alpha=0.7,
                           label=f'Product {product+1}', bins=20)
            axes[0].set_title('Heuristic Policy - FG Actions')
            axes[0].set_xlabel('Order Quantity')
            axes[0].set_ylabel('Frequency')
            axes[0].legend()
            axes[0].grid(True, alpha=0.3)

            for product in range(num_products):
                axes[1].hist(trained_fg[:, product], alpha=0.7,
                           label=f'Product {product+1}', bins=20)
            axes[1].set_title('Trained Policy - FG Actions')
            axes[1].set_xlabel('Order Quantity')
            axes[1].set_ylabel('Frequency')
            axes[1].legend()
            axes[1].grid(True, alpha=0.3)

    else:
        # RM agent actions (single dimension)
        heuristic_rm = np.array(heuristic_actions['rm_agent']).flatten()
        trained_rm = np.array(trained_actions['rm_agent']).flatten()

        axes[0].hist(heuristic_rm, bins=30, alpha=0.7, color='red', label='Heuristic')
        axes[0].set_title('Heuristic Policy - RM Actions')
        axes[0].set_xlabel('Order Quantity')
        axes[0].set_ylabel('Frequency')
        axes[0].grid(True, alpha=0.3)

        axes[1].hist(trained_rm, bins=30, alpha=0.7, color='blue', label='Trained')
        axes[1].set_title('Trained Policy - RM Actions')
        axes[1].set_xlabel('Order Quantity')
        axes[1].set_ylabel('Frequency')
        axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

def create_comprehensive_evaluation(env,
                                  fg_agent,
                                  rm_agent,
                                  heuristic_policy,
                                  num_eval_episodes: int = 5,
                                  steps_per_episode: int = 100):
    """
    Run comprehensive evaluation comparing trained policy vs heuristic

    Args:
        env: Inventory environment
        fg_agent: Trained FG agent
        rm_agent: Trained RM agent
        heuristic_policy: Heuristic policy function
        num_eval_episodes: Number of evaluation episodes
        steps_per_episode: Steps per evaluation episode
    """
    print("🚀 RUNNING COMPREHENSIVE EVALUATION")
    print("="*60)

    # Store results
    trained_results = []
    heuristic_results = []

    trained_actions = {'fg_agent': [], 'rm_agent': []}
    heuristic_actions = {'fg_agent': [], 'rm_agent': []}

    # Evaluate trained policy
    print("Evaluating Trained Policy...")
    for episode in range(num_eval_episodes):
        obs = env.reset()
        episode_history = []

        for step in range(steps_per_episode):
            # Trained policy
            fg_action, _, _ = fg_agent.act(obs['fg_agent'], deterministic=True)
            rm_action, _, _ = rm_agent.act(obs['rm_agent'], deterministic=True)

            trained_actions['fg_agent'].append(fg_action)
            trained_actions['rm_agent'].append(rm_action)

            joint_action = {'fg_agent': fg_action, 'rm_agent': rm_action}
            obs, _, done, info = env.step(joint_action)
            episode_history.append(info)

            if done:
                break

        trained_results.extend(episode_history)

    # Evaluate heuristic policy
    print("Evaluating Heuristic Policy...")
    for episode in range(num_eval_episodes):
        obs = env.reset()
        episode_history = []

        for step in range(steps_per_episode):
            # Heuristic policy
            heuristic_action = heuristic_policy(obs)

            heuristic_actions['fg_agent'].append(heuristic_action['fg_agent'])
            heuristic_actions['rm_agent'].append(heuristic_action['rm_agent'])

            obs, _, done, info = env.step(heuristic_action)
            episode_history.append(info)

            if done:
                break

        heuristic_results.extend(episode_history)

    # Generate comprehensive visualizations
    print("\n📊 GENERATING VISUALIZATIONS...")

    # 1. Policy comparison
    plot_agent_behavior_comparison(heuristic_results, trained_results)

    # 2. Action distributions
    plot_action_distribution(heuristic_actions, trained_actions, "fg_agent")
    plot_action_distribution(heuristic_actions, trained_actions, "rm_agent")

    # 3. Detailed trajectory analysis (first episode)
    plot_environment_run(heuristic_results[:steps_per_episode], "Heuristic Policy - Detailed Trajectory")
    plot_environment_run(trained_results[:steps_per_episode], "Trained Policy - Detailed Trajectory")

    return trained_results, heuristic_results

# Example usage function
def run_quick_evaluation(env, fg_agent, rm_agent):
    """
    Quick evaluation function for trained agents
    """
    from utils.heuristic_policy import get_heuristic_action

    print("🔍 RUNNING QUICK EVALUATION")
    print("="*50)

    # Run a single episode with trained policy
    obs = env.reset()
    trained_history = []

    for step in range(200):  # Shorter evaluation
        fg_action, _, _ = fg_agent.act(obs['fg_agent'], deterministic=True)
        rm_action, _, _ = rm_agent.act(obs['rm_agent'], deterministic=True)

        joint_action = {'fg_agent': fg_action, 'rm_agent': rm_action}
        obs, reward, done, info = env.step(joint_action)
        trained_history.append(info)

        if done:
            break

    # Run same episode with heuristic
    obs = env.reset()
    heuristic_history = []

    for step in range(200):
        heuristic_action = get_heuristic_action(obs)
        obs, reward, done, info = env.step(heuristic_action)
        heuristic_history.append(info)

        if done:
            break

    # Compare policies
    plot_agent_behavior_comparison(heuristic_history, trained_history)

    return trained_history, heuristic_history

if __name__ == "__main__":
    # Test the evaluation functions
    from environment.inventory_env import InventoryMAMDPEnv
    from utils.heuristic_policy import get_heuristic_action

    env = InventoryMAMDPEnv()

    # Create dummy agents for testing (you would replace with your trained agents)
    from models.networks import FGActorCritic, RMActorCritic
    from agents.ppo_agent import PPOAgent

    fg_network = FGActorCritic(
        state_dim=env.observation_space['fg_agent'].shape[0],
        action_dim=env.action_space_fg.shape[0]
    )
    rm_network = RMActorCritic(
        state_dim=env.observation_space['rm_agent'].shape[0],
        action_dim=env.action_space_rm.shape[0]
    )

    fg_agent = PPOAgent(fg_network, agent_type="fg_agent")
    rm_agent = PPOAgent(rm_network, agent_type="rm_agent")

    # Run quick evaluation
    trained_history, heuristic_history = run_quick_evaluation(env, fg_agent, rm_agent)

In [ ]:
%%writefile main_training.py
"""
Complete training script with all critical fixes implemented
INCLUDES: Enhanced FG observation, Curriculum Learning, and Exploration Scheduling
"""

import numpy as np
import torch
import time
import os
import random
from typing import Dict, List, Tuple
import matplotlib.pyplot as plt
from collections import deque

from environment.inventory_env import InventoryMAMDPEnv
from models.networks import FGActorCritic, RMActorCritic
from agents.ppo_agent import PPOAgent
from utils.heuristic_policy import get_heuristic_action
import configs.config as config
from utils.evaluation import (
    plot_training_progress,
    run_quick_evaluation,
    create_comprehensive_evaluation
)

# Set random seeds for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

class TrainingLogger:
    """Enhanced logger for training system with cost component tracking"""

    def __init__(self, log_dir: str = "/content/logs"):
        os.makedirs(log_dir, exist_ok=True)
        self.log_dir = log_dir

        # Enhanced data storage with cost components
        self.episode_data = {
            'total_reward': [],
            'total_cost': [],
            'total_lost_sales': [],
            'total_rm_stockouts': [],
            'episode_length': [],
            # ADD cost components
            'fg_holding_cost': [],
            'fg_shortage_cost': [],
            'rm_holding_cost': [],
            'rm_stockout_penalty': [],
            'rm_order_cost': [],
            'mean_fg_action': [],
            'std_fg_action': [],
            'mean_rm_action': [],
            'std_rm_action': [],
            'max_fg_action': [],
            'max_rm_action': [],
        }

    def log_episode(self, episode: int, stats: Dict):
        """Log episode statistics with cost components"""
        # Store basic stats
        for key in self.episode_data:
            if key in stats:
                self.episode_data[key].append(stats[key])

        if episode % 10 == 0:
            print(f"Episode {episode:4d} | "
                  f"Reward: {stats.get('total_reward', 0):8.2f} | "
                  f"Cost: {stats.get('total_cost', 0):8.2f} | "
                  f"Lost Sales: {stats.get('total_lost_sales', 0):6.1f} | "
                  f"RM Stockouts: {stats.get('total_rm_stockouts', 0):4d}")

            # ADD cost component logging for debugging
            if episode % 50 == 0 and 'fg_holding_cost' in stats:
                print(f"  Cost Components - FG Hold: {stats['fg_holding_cost']:.1f}, "
                      f"FG Short: {stats['fg_shortage_cost']:.1f}, "
                      f"RM Hold: {stats['rm_holding_cost']:.1f}, "
                      f"RM Penalty: {stats['rm_stockout_penalty']:.1f}")

    def plot_progress(self):
        """Plot training progress using the new evaluation functions"""
        print("\n📈 GENERATING TRAINING ANALYSIS...")
        plot_training_progress(
            [dict(zip(self.episode_data.keys(), values))
             for values in zip(*self.episode_data.values())],
            window_size=10,
            title="Multi-Agent Training Progress"
        )

def collect_expert_demonstrations(env, num_demos: int = 2000) -> dict:
    """Collect expert demonstrations using simplified heuristic"""
    print(f"Collecting {num_demos} expert demonstrations...")

    demonstrations = {
        'fg_states': [],
        'fg_actions': [],
        'rm_states': [],
        'rm_actions': []
    }

    obs = env.reset()
    demo_count = 0

    while demo_count < num_demos:
        # Get expert action from simplified heuristic policy
        expert_actions = get_heuristic_action(obs)

        # Store demonstration
        demonstrations['fg_states'].append(obs['fg_agent'])
        demonstrations['fg_actions'].append(expert_actions['fg_agent'])
        demonstrations['rm_states'].append(obs['rm_agent'])
        demonstrations['rm_actions'].append(expert_actions['rm_agent'])
        demo_count += 1

        # Step environment
        obs, _, done, _ = env.step(expert_actions)

        if done:
            obs = env.reset()

        if demo_count % 500 == 0:
            print(f" Collected {demo_count}/{num_demos} demonstrations")

    # Convert to arrays
    for key in demonstrations:
        demonstrations[key] = np.array(demonstrations[key])

    print(f"✓ Collected {num_demos} expert demonstrations")
    fg_mean = np.mean(demonstrations['fg_actions'], axis=0)
    fg_std = np.std(demonstrations['fg_actions'], axis=0)
    print(f"FG Actions - Mean: {np.round(fg_mean, 2)}, Std: {np.round(fg_std, 2)}")
    rm_mean = np.mean(demonstrations['rm_actions'], axis=0)
    rm_std = np.std(demonstrations['rm_actions'], axis=0)
    print(f"RM Actions - Mean: {np.round(rm_mean, 2)}, Std: {np.round(rm_std, 2)}")

    return demonstrations

def pretrain_networks(fg_network, rm_network, demonstrations, epochs: int = 100):
    """Pretrain networks using behavioral cloning"""
    print("Pretraining networks with expert demonstrations...")

    # Convert to tensors
    fg_states_tensor = torch.FloatTensor(demonstrations['fg_states'])
    fg_actions_tensor = torch.FloatTensor(demonstrations['fg_actions'])
    rm_states_tensor = torch.FloatTensor(demonstrations['rm_states'])
    rm_actions_tensor = torch.FloatTensor(demonstrations['rm_actions'])

    # FG Network pretraining
    fg_optimizer = torch.optim.Adam(fg_network.parameters(), lr=1e-3)
    fg_criterion = torch.nn.MSELoss()

    fg_losses = []
    for epoch in range(epochs):
        fg_optimizer.zero_grad()

        mu, _, _ = fg_network(fg_states_tensor)
        fg_loss = fg_criterion(mu, fg_actions_tensor)
        fg_loss.backward()
        fg_optimizer.step()
        fg_losses.append(fg_loss.item())

        if epoch % 20 == 0:
            print(f" FG Epoch {epoch:3d}: loss = {fg_loss.item():.4f}")

    # RM Network pretraining
    rm_optimizer = torch.optim.Adam(rm_network.parameters(), lr=1e-3)
    rm_criterion = torch.nn.MSELoss()

    rm_losses = []
    for epoch in range(epochs):
        rm_optimizer.zero_grad()
        mu, _, _ = rm_network(rm_states_tensor)
        rm_loss = rm_criterion(mu, rm_actions_tensor)
        rm_loss.backward()
        rm_optimizer.step()
        rm_losses.append(rm_loss.item())

        if epoch % 20 == 0:
            print(f" RM Epoch {epoch:3d}: loss = {rm_loss.item():.4f}")

    # Test final performance
    with torch.no_grad():
        fg_mu, _, _ = fg_network(fg_states_tensor[:100])
        rm_mu, _, _ = rm_network(rm_states_tensor[:100])

        fg_error = torch.mean(torch.abs(fg_mu - fg_actions_tensor[:100])).item()
        rm_error = torch.mean(torch.abs(rm_mu - rm_actions_tensor[:100])).item()

    print(f"✓ Pretraining completed")
    print(f" FG Final action error: {fg_error:.4f}")
    print(f" RM Final action error: {rm_error:.4f}")

    return fg_network, rm_network

def run_training_episode(env, fg_agent, rm_agent, episode: int,
                        exploration: bool = True) -> Dict:
    """Run a single training episode with the system"""
    obs = env.reset()
    episode_reward = 0
    episode_length = 0
    episode_costs = []
    episode_lost_sales = []
    episode_rm_stockouts = []
    episode_fg_holding = []
    episode_fg_shortage = []
    episode_rm_holding = []
    episode_rm_penalty = []
    episode_rm_order = []
    episode_fg_actions = []
    episode_rm_actions = []


    while True:
        # Get actions from both agents
        fg_action, fg_log_prob, fg_value = fg_agent.act(
            obs['fg_agent'], deterministic=not exploration
        )
        rm_action, rm_log_prob, rm_value = rm_agent.act(
            obs['rm_agent'], deterministic=not exploration
        )

        # Store actions for analysis
        episode_fg_actions.append(fg_action)
        episode_rm_actions.append(rm_action)

        joint_action = {
            'fg_agent': fg_action,
            'rm_agent': rm_action
        }

        # Take environment step
        next_obs, rewards, done, info = env.step(joint_action)

        # Store transitions (both agents get same reward due to unified reward)
        fg_agent.store_transition(
            obs['fg_agent'], fg_action, fg_log_prob,
            rewards['fg_agent'], fg_value, done
        )
        rm_agent.store_transition(
            obs['rm_agent'], rm_action, rm_log_prob,
            rewards['rm_agent'], rm_value, done
        )

        # Update statistics
        episode_reward += rewards['fg_agent'] # Same for both agents
        episode_length += 1
        episode_costs.append(info['total_cost'])
        episode_lost_sales.append(np.sum(info['lost_sales']))
        episode_rm_stockouts.append(info['rm_stockout_count'])
        episode_fg_holding.append(info.get('fg_holding_cost', 0))
        episode_fg_shortage.append(info.get('fg_shortage_cost', 0))
        episode_rm_holding.append(info.get('rm_holding_cost', 0))
        episode_rm_penalty.append(info.get('rm_stockout_penalty', 0))
        episode_rm_order.append(info.get('rm_order_cost', 0))

        # Update observation
        obs = next_obs

        # Perform PPO updates if enough data collected
        if episode_length % config.config.training.UPDATE_TIMESTEPS == 0:
            fg_stats = fg_agent.update()
            rm_stats = rm_agent.update()

            # Log training stats if needed
            if 'policy_loss' in fg_stats:
                print(f"    FG Update - Policy: {fg_stats['policy_loss']:.4f}, "
                      f"Value: {fg_stats['value_loss']:.4f}, "
                      f"Mono: {fg_stats.get('monotonicity_penalty', 0):.6f}")

        if done:
            break

    # Calculate episode statistics
    stats = {
        'total_reward': episode_reward,
        'total_cost': np.sum(episode_costs),
        'total_lost_sales': np.sum(episode_lost_sales),
        'total_rm_stockouts': np.sum(episode_rm_stockouts),
        'episode_length': episode_length,
        'fg_holding_cost': np.sum(episode_fg_holding),
        'fg_shortage_cost': np.sum(episode_fg_shortage),
        'rm_holding_cost': np.sum(episode_rm_holding),
        'rm_stockout_penalty': np.sum(episode_rm_penalty),
        'rm_order_cost': np.sum(episode_rm_order)
    }

    # Calculate action statistics
    if episode_fg_actions:
        fg_actions_array = np.array(episode_fg_actions)
        rm_actions_array = np.array(episode_rm_actions)

        stats.update({
            'mean_fg_action': np.mean(fg_actions_array),
            'std_fg_action': np.std(fg_actions_array),
            'mean_rm_action': np.mean(rm_actions_array),
            'std_rm_action': np.std(rm_actions_array),
            'max_fg_action': np.max(fg_actions_array),
            'max_rm_action': np.max(rm_actions_array),
        })

    return stats

def evaluate_policy(env, fg_agent, rm_agent, num_episodes: int = 5) -> Dict:
    """Evaluate the trained policy"""
    print(f"Evaluating policy over {num_episodes} episodes...")

    eval_returns = []
    eval_costs = []
    eval_lost_sales = []
    eval_rm_stockouts = []

    for eval_ep in range(num_episodes):
        obs = env.reset()
        episode_return = 0
        episode_costs = []
        episode_lost_sales = []
        episode_rm_stockouts = []

        while True:
            # Use deterministic actions for evaluation
            fg_action, _, _ = fg_agent.act(obs['fg_agent'], deterministic=True)
            rm_action, _, _ = rm_agent.act(obs['rm_agent'], deterministic=True)

            joint_action = {
                'fg_agent': fg_action,
                'rm_agent': rm_action
            }

            obs, rewards, done, info = env.step(joint_action)

            episode_return += rewards['fg_agent'] # Same for both agents
            episode_costs.append(info['total_cost'])
            episode_lost_sales.append(np.sum(info['lost_sales']))
            episode_rm_stockouts.append(info['rm_stockout_count'])

            if done:
                break

        eval_returns.append(episode_return)
        eval_costs.append(np.sum(episode_costs))
        eval_lost_sales.append(np.sum(episode_lost_sales))
        eval_rm_stockouts.append(np.sum(episode_rm_stockouts))

    eval_stats = {
        'mean_return': np.mean(eval_returns),
        'std_return': np.std(eval_returns),
        'mean_cost': np.mean(eval_costs),
        'mean_lost_sales': np.mean(eval_lost_sales),
        'mean_rm_stockouts': np.mean(eval_rm_stockouts)
    }

    print(f" Mean Return: {eval_stats['mean_return']:8.2f} ± {eval_stats['std_return']:5.2f}")
    print(f" Mean Cost: {eval_stats['mean_cost']:8.2f}")
    print(f" Mean Lost Sales: {eval_stats['mean_lost_sales']:6.1f}")
    print(f" Mean RM Stockouts: {eval_stats['mean_rm_stockouts']:4.1f}")
    return eval_stats

def get_exploration_schedule(current_episode: int, total_episodes: int,
                           exploration_phase_ratio: float = 0.7) -> bool:
    """
    Determine exploration strategy based on training phase
    Returns: True if we should explore, False for exploitation
    """
    exploration_threshold = int(total_episodes * exploration_phase_ratio)

    # Explore more in early phases, exploit more later
    if current_episode < exploration_threshold:
        # Early phase: explore with 80% probability
        return random.random() < 0.8
    else:
        # Late phase: explore with only 20% probability
        return random.random() < 0.2

def compare_agent_actions(env, fg_agent, rm_agent, heuristic_policy, num_steps: int = 200):
    """
    Compare actions between heuristic policy and trained PPO agent

    Args:
        env: Inventory environment
        fg_agent: Trained FG PPO agent
        rm_agent: Trained RM PPO agent
        heuristic_policy: Heuristic policy function
        num_steps: Number of steps to compare
    """

    print("=" * 80)
    print("ACTION COMPARISON: Heuristic vs Trained PPO Agent")
    print("=" * 80)

    # Reset environment for both policies (same initial state)
    env.seed(42)  # For reproducibility
    obs_heuristic = env.reset()
    obs_ppo = env.reset()

    # Store actions for comparison
    heuristic_actions = {
        'fg': [],
        'rm': []
    }

    ppo_actions = {
        'fg': [],
        'rm': []
    }

    # Store states to understand context
    states = {
        'fg_inventory': [],
        'rm_inventory': [],
        'demand': []
    }

    # Run both policies for the same number of steps
    for step in range(num_steps):
        # Heuristic policy actions
        heuristic_action = heuristic_policy(obs_heuristic)
        heuristic_actions['fg'].append(heuristic_action['fg_agent'])
        heuristic_actions['rm'].append(heuristic_action['rm_agent'])

        # PPO agent actions
        fg_action_ppo, _, _ = fg_agent.act(obs_ppo['fg_agent'], deterministic=True)
        rm_action_ppo, _, _ = rm_agent.act(obs_ppo['rm_agent'], deterministic=True)
        ppo_actions['fg'].append(fg_action_ppo)
        ppo_actions['rm'].append(rm_action_ppo)

        # Store current state for context
        states['fg_inventory'].append(obs_ppo['fg_agent'][:5])  # First 5 elements are FG inventory
        states['rm_inventory'].append(obs_ppo['rm_agent'][0])   # First element is RM inventory

        # Step both environments
        obs_heuristic, _, done_heuristic, _ = env.step(heuristic_action)
        obs_ppo, _, done_ppo, _ = env.step({
            'fg_agent': fg_action_ppo,
            'rm_agent': rm_action_ppo
        })

        if done_heuristic or done_ppo:
            obs_heuristic = env.reset()
            obs_ppo = env.reset()
            print(f"Environment reset at step {step}")

    # Convert to numpy arrays
    heuristic_fg = np.array(heuristic_actions['fg'])
    heuristic_rm = np.array(heuristic_actions['rm'])
    ppo_fg = np.array(ppo_actions['fg'])
    ppo_rm = np.array(ppo_actions['rm'])

    # Create comprehensive comparison plots
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle('Action Comparison: Heuristic Policy vs Trained PPO Agent',
                 fontsize=16, fontweight='bold')

    # Plot 1: FG Actions over time (first product)
    product_idx = 0  # Compare first product
    axes[0, 0].plot(heuristic_fg[:, product_idx], 'r-', linewidth=2,
                   label='Heuristic', alpha=0.8)
    axes[0, 0].plot(ppo_fg[:, product_idx], 'b-', linewidth=2,
                   label='PPO Agent', alpha=0.8)
    axes[0, 0].set_title(f'FG Actions - Product {product_idx + 1}')
    axes[0, 0].set_xlabel('Time Step')
    axes[0, 0].set_ylabel('Order Quantity')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

    # Plot 2: FG Action distribution (all products)
    all_heuristic_fg = heuristic_fg.flatten()
    all_ppo_fg = ppo_fg.flatten()

    axes[0, 1].hist(all_heuristic_fg, bins=30, alpha=0.7, color='red',
                   label='Heuristic', density=True)
    axes[0, 1].hist(all_ppo_fg, bins=30, alpha=0.7, color='blue',
                   label='PPO Agent', density=True)
    axes[0, 1].set_title('FG Action Distribution (All Products)')
    axes[0, 1].set_xlabel('Order Quantity')
    axes[0, 1].set_ylabel('Density')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)

    # Plot 3: FG Action statistics by product
    products = range(heuristic_fg.shape[1])
    heuristic_means = np.mean(heuristic_fg, axis=0)
    ppo_means = np.mean(ppo_fg, axis=0)

    x_pos = np.arange(len(products))
    width = 0.35

    axes[0, 2].bar(x_pos - width/2, heuristic_means, width,
                   label='Heuristic', color='red', alpha=0.8)
    axes[0, 2].bar(x_pos + width/2, ppo_means, width,
                   label='PPO Agent', color='blue', alpha=0.8)
    axes[0, 2].set_title('Average FG Actions by Product')
    axes[0, 2].set_xlabel('Product')
    axes[0, 2].set_ylabel('Average Order Quantity')
    axes[0, 2].set_xticks(x_pos)
    axes[0, 2].set_xticklabels([f'P{i+1}' for i in products])
    axes[0, 2].legend()
    axes[0, 2].grid(True, alpha=0.3)

    # Plot 4: RM Actions over time
    axes[1, 0].plot(heuristic_rm.flatten(), 'r-', linewidth=2,
                   label='Heuristic', alpha=0.8)
    axes[1, 0].plot(ppo_rm.flatten(), 'b-', linewidth=2,
                   label='PPO Agent', alpha=0.8)
    axes[1, 0].set_title('RM Actions Over Time')
    axes[1, 0].set_xlabel('Time Step')
    axes[1, 0].set_ylabel('Order Quantity')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)

    # Plot 5: RM Action distribution
    axes[1, 1].hist(heuristic_rm.flatten(), bins=30, alpha=0.7, color='red',
                   label='Heuristic', density=True)
    axes[1, 1].hist(ppo_rm.flatten(), bins=30, alpha=0.7, color='blue',
                   label='PPO Agent', density=True)
    axes[1, 1].set_title('RM Action Distribution')
    axes[1, 1].set_xlabel('Order Quantity')
    axes[1, 1].set_ylabel('Density')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)

    # Plot 6: Action correlation with inventory
    rm_inventory = np.array(states['rm_inventory'])

    # Scatter plot: RM Action vs RM Inventory
    scatter = axes[1, 2].scatter(rm_inventory[:len(heuristic_rm)],
                                heuristic_rm.flatten(),
                                c='red', alpha=0.6, label='Heuristic', s=30)
    scatter = axes[1, 2].scatter(rm_inventory[:len(ppo_rm)],
                                ppo_rm.flatten(),
                                c='blue', alpha=0.6, label='PPO Agent', s=30)
    axes[1, 2].set_title('RM Action vs RM Inventory')
    axes[1, 2].set_xlabel('RM Inventory Level')
    axes[1, 2].set_ylabel('RM Order Quantity')
    axes[1, 2].legend()
    axes[1, 2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Print detailed statistics
    print("\n" + "=" * 60)
    print("ACTION STATISTICS COMPARISON")
    print("=" * 60)

    # FG Action statistics
    print("\nFINISHED GOODS ACTIONS:")
    print("-" * 40)
    for i in range(heuristic_fg.shape[1]):
        print(f"Product {i+1}:")
        print(f"  Heuristic - Mean: {heuristic_means[i]:.2f}, "
              f"Std: {np.std(heuristic_fg[:, i]):.2f}, "
              f"Range: [{np.min(heuristic_fg[:, i]):.2f}, {np.max(heuristic_fg[:, i]):.2f}]")
        print(f"  PPO Agent - Mean: {ppo_means[i]:.2f}, "
              f"Std: {np.std(ppo_fg[:, i]):.2f}, "
              f"Range: [{np.min(ppo_fg[:, i]):.2f}, {np.max(ppo_fg[:, i]):.2f}]")
        difference = ppo_means[i] - heuristic_means[i]
        print(f"  Difference: {difference:+.2f} ({difference/heuristic_means[i]*100:+.1f}%)\n")

    # RM Action statistics
    print("\nRAW MATERIAL ACTIONS:")
    print("-" * 40)
    heuristic_rm_mean = np.mean(heuristic_rm)
    ppo_rm_mean = np.mean(ppo_rm)
    print(f"Heuristic - Mean: {heuristic_rm_mean:.2f}, "
          f"Std: {np.std(heuristic_rm):.2f}, "
          f"Range: [{np.min(heuristic_rm):.2f}, {np.max(heuristic_rm):.2f}]")
    print(f"PPO Agent - Mean: {ppo_rm_mean:.2f}, "
          f"Std: {np.std(ppo_rm):.2f}, "
          f"Range: [{np.min(ppo_rm):.2f}, {np.max(ppo_rm):.2f}]")
    rm_difference = ppo_rm_mean - heuristic_rm_mean
    print(f"Difference: {rm_difference:+.2f} ({rm_difference/heuristic_rm_mean*100:+.1f}%)")

    # Action variability comparison
    print("\nACTION VARIABILITY:")
    print("-" * 40)
    fg_variability_heuristic = np.mean(np.std(heuristic_fg, axis=0))
    fg_variability_ppo = np.mean(np.std(ppo_fg, axis=0))
    rm_variability_heuristic = np.std(heuristic_rm)
    rm_variability_ppo = np.std(ppo_rm)

    print(f"FG Action Variability:")
    print(f"  Heuristic: {fg_variability_heuristic:.3f}")
    print(f"  PPO Agent: {fg_variability_ppo:.3f}")
    print(f"  Ratio: {fg_variability_ppo/fg_variability_heuristic:.3f}")

    print(f"RM Action Variability:")
    print(f"  Heuristic: {rm_variability_heuristic:.3f}")
    print(f"  PPO Agent: {rm_variability_ppo:.3f}")
    print(f"  Ratio: {rm_variability_ppo/rm_variability_heuristic:.3f}")

    return {
        'heuristic_actions': heuristic_actions,
        'ppo_actions': ppo_actions,
        'states': states
    }

# Add this to your main_training.py or evaluation script

def enhanced_comprehensive_evaluation(env, fg_agent, rm_agent, heuristic_policy,
                                    num_eval_episodes: int = 5, steps_per_episode: int = 100):
    """
    Enhanced evaluation with action comparison
    """
    print("#" * 80)
    print("ENHANCED COMPREHENSIVE EVALUATION")
    print("#" * 80)

    # 1. Run action comparison
    print("\n[1/3] RUNNING ACTION COMPARISON...")
    action_comparison = compare_agent_actions(
        env, fg_agent, rm_agent, heuristic_policy, num_steps=steps_per_episode
    )

    # 2. Run performance comparison (your existing function)
    print("\n[2/3] RUNNING PERFORMANCE COMPARISON...")
    trained_results, heuristic_results = create_comprehensive_evaluation(
        env, fg_agent, rm_agent, heuristic_policy,
        num_eval_episodes=num_eval_episodes,
        steps_per_episode=steps_per_episode
    )

    # 3. Additional analysis: Action patterns in different inventory states
    print("\n[3/3] ANALYZING ACTION PATTERNS...")
    analyze_action_patterns(action_comparison)

    return trained_results, heuristic_results, action_comparison

def analyze_action_patterns(action_comparison):
    """
    Analyze how actions differ in different inventory states
    """
    states = action_comparison['states']
    heuristic_actions = action_comparison['heuristic_actions']
    ppo_actions = action_comparison['ppo_actions']

    rm_inventory = np.array(states['rm_inventory'])
    heuristic_rm = np.array(heuristic_actions['rm']).flatten()
    ppo_rm = np.array(ppo_actions['rm']).flatten()

    # Analyze low inventory vs high inventory behavior
    inventory_threshold = np.median(rm_inventory)

    low_inv_mask = rm_inventory < inventory_threshold
    high_inv_mask = rm_inventory >= inventory_threshold

    print("\n" + "=" * 50)
    print("ACTION PATTERN ANALYSIS BY INVENTORY LEVEL")
    print("=" * 50)

    print(f"\nLow RM Inventory (< {inventory_threshold:.1f}):")
    print(f"  Heuristic RM Orders: {np.mean(heuristic_rm[low_inv_mask]):.2f}")
    print(f"  PPO Agent RM Orders: {np.mean(ppo_rm[low_inv_mask]):.2f}")

    print(f"\nHigh RM Inventory (>= {inventory_threshold:.1f}):")
    print(f"  Heuristic RM Orders: {np.mean(heuristic_rm[high_inv_mask]):.2f}")
    print(f"  PPO Agent RM Orders: {np.mean(ppo_rm[high_inv_mask]):.2f}")

def evaluate_trained_models():
    """Standalone function to evaluate trained models"""
    print("🧪 EVALUATING TRAINED MODELS")

    from environment.inventory_env import InventoryMAMDPEnv
    from models.networks import FGActorCritic, RMActorCritic
    from agents.ppo_agent import PPOAgent
    from utils.heuristic_policy import get_heuristic_action
    from utils.evaluation import create_comprehensive_evaluation

    # Initialize environment and agents
    env = InventoryMAMDPEnv()

    fg_network = FGActorCritic(
        state_dim=env.observation_space['fg_agent'].shape[0],
        action_dim=env.action_space_fg.shape[0]
    )
    rm_network = RMActorCritic(
        state_dim=env.observation_space['rm_agent'].shape[0],
        action_dim=env.action_space_rm.shape[0]
    )

    fg_agent = PPOAgent(fg_network, agent_type="fg_agent")
    rm_agent = PPOAgent(rm_network, agent_type="rm_agent")

    # Load trained models
    try:
        fg_agent.load_checkpoint("/content/checkpoints/best_fg_model.pth")
        rm_agent.load_checkpoint("/content/checkpoints/best_rm_model.pth")
        print("✓ Loaded trained models")
    except Exception as e:
        print(f"⚠️  Could not load models: {e}")
        print("⚠️  Using untrained models for evaluation")

    # Run comprehensive evaluation
    trained_results, heuristic_results = create_comprehensive_evaluation(
        env, fg_agent, rm_agent, get_heuristic_action,
        num_eval_episodes=5, steps_per_episode=200
    )

    return trained_results, heuristic_results

def main_training():
    """Main training function with all fixes implemented"""
    print("=== STARTING SYSTEM TRAINING ===")

    # Setup
    env = InventoryMAMDPEnv()
    logger = TrainingLogger()

    # Get environment dimensions
    fg_state_dim = env.observation_space['fg_agent'].shape[0]
    fg_action_dim = env.action_space_fg.shape[0]
    rm_state_dim = env.observation_space['rm_agent'].shape[0]
    rm_action_dim = env.action_space_rm.shape[0]

    print(f"\nEnvironment setup:")
    print(f" FG: state_dim={fg_state_dim}, action_dim={fg_action_dim}")
    print(f" RM: state_dim={rm_state_dim}, action_dim={rm_action_dim}")

    # Create networks
    fg_network = FGActorCritic(fg_state_dim, fg_action_dim, output_multiplier=30)
    rm_network = RMActorCritic(rm_state_dim, rm_action_dim)

    # Step 1: Collect expert demonstrations
    demonstrations = collect_expert_demonstrations(env, num_demos=2000)

    # Step 2: Pretrain networks
    fg_network, rm_network = pretrain_networks(
        fg_network, rm_network, demonstrations, epochs=120
    )

    # Step 3: Create agents with pretrained networks
    fg_agent = PPOAgent(fg_network, agent_type="fg_agent")
    rm_agent = PPOAgent(rm_network, agent_type="rm_agent")

    # Evaluate before training
    # run_quick_evaluation(env, fg_agent, rm_agent)
    create_comprehensive_evaluation(
        env, fg_agent, rm_agent, get_heuristic_action,
        num_eval_episodes=3, steps_per_episode=200
    )

    # TRAINING PROTOCOL
    print("\n" + "="*60)
    print("TRAINING AFTER PRE-TRAINING")
    print("="*60)

    # Reset learning rates to original values (in case they were modified)
    for param_group in fg_agent.actor_optimizer.param_groups:
        param_group['lr'] = config.config.model.LEARNING_RATE_ACTOR
    for param_group in rm_agent.actor_optimizer.param_groups:
        param_group['lr'] = config.config.model.LEARNING_RATE_ACTOR

    print("✓ Both agents ready for joint training with original learning rates")

    # Training parameters for joint phase
    joint_episodes = 1000
    eval_interval = 20

    # Track best performance
    best_eval_return = -np.inf
    best_episode = 0

    # Training loop for joint fine-tuning
    start_time = time.time()

    for episode in range(joint_episodes):
        # Use exploration scheduling
        exploration = get_exploration_schedule(episode, joint_episodes, exploration_phase_ratio=0.7)

        episode_stats = run_training_episode(
            env, fg_agent, rm_agent, episode, exploration=exploration
        )

        # Log episode (offset by previous phases)
        logger.log_episode(episode, episode_stats)

        # Evaluation
        if episode % 50 == 0 and episode > 0:
            print(f"\nPERIODIC EVALUATION at Joint Episode {episode}")
            print("="*50)

            try:
                # Run quick evaluation
                trained_history, heuristic_history = run_quick_evaluation(env, fg_agent, rm_agent)

                # Save evaluation snapshot
                eval_dir = f"/content/evaluation_snapshots/episode_{episode}"
                os.makedirs(eval_dir, exist_ok=True)
                print(f"Evaluation snapshot saved to {eval_dir}")

            except Exception as e:
                print(f"⚠️  Evaluation failed: {e}")

        if episode % eval_interval == 0 and episode > 0:
            print(f"\n--- Evaluation at Joint Episode {episode} (Total: {episode}) ---")
            eval_stats = evaluate_policy(env, fg_agent, rm_agent, num_episodes=3)

            # Save best model
            if eval_stats['mean_return'] > best_eval_return:
                best_eval_return = eval_stats['mean_return']
                best_episode = episode

                # Save checkpoints
                os.makedirs("/content/checkpoints", exist_ok=True)
                fg_agent.save_checkpoint("/content/checkpoints/best_fg_model.pth")
                rm_agent.save_checkpoint("/content/checkpoints/best_rm_model.pth")
                print(f"   💾 New best model saved (return: {best_eval_return:.2f})")

            print("-" * 50)

    # Training completed
    training_time = time.time() - start_time
    print(f"\n🎉 TRAINING COMPLETED ===")
    print(f"Total training time: {training_time:.1f}s ({training_time/60:.1f} minutes)")
    print(f"Total episodes: {300 + joint_episodes} (150 RM + 150 FG + {joint_episodes} Joint)")
    print(f"Best evaluation return: {best_eval_return:.2f} (episode {best_episode})")

    # 🆕 COMPREHENSIVE FINAL EVALUATION
    print(f"\n📊 COMPREHENSIVE FINAL EVALUATION ===")

    try:
        # Load best models for final evaluation
        if os.path.exists("/content/checkpoints/best_fg_model.pth"):
            fg_agent.load_checkpoint("/content/checkpoints/best_fg_model.pth")
            rm_agent.load_checkpoint("/content/checkpoints/best_rm_model.pth")
            print("✓ Loaded best models for final evaluation")

        # Run comprehensive evaluation
        trained_results, heuristic_results = create_comprehensive_evaluation(
            env, fg_agent, rm_agent, get_heuristic_action,
            num_eval_episodes=3, steps_per_episode=200
        )

    except Exception as e:
        print(f"⚠️  Comprehensive evaluation failed: {e}")
        # Fallback to simple evaluation
        final_eval = evaluate_policy(env, fg_agent, rm_agent, num_episodes=10)

    # Compare with heuristic baseline
    print(f"\n📈 HEURISTIC BASELINE COMPARISON ===")
    heuristic_returns = []
    for _ in range(10):
        obs = env.reset()
        heuristic_return = 0
        while True:
            action = get_heuristic_action(obs)
            obs, rewards, done, _ = env.step(action)
            heuristic_return += rewards['fg_agent']
            if done:
                break
        heuristic_returns.append(heuristic_return)

    heuristic_mean = np.mean(heuristic_returns)
    heuristic_std = np.std(heuristic_returns)

    # Use final evaluation results
    if 'final_eval' in locals():
        trained_mean = final_eval['mean_return']
        trained_std = final_eval['std_return']
    else:
        # Estimate from training
        trained_mean = best_eval_return
        trained_std = 0

    print(f"Heuristic Mean Return: {heuristic_mean:8.2f} ± {heuristic_std:5.2f}")
    print(f"Trained Policy Mean Return: {trained_mean:8.2f} ± {trained_std:5.2f}")
    improvement = ((trained_mean - heuristic_mean) / abs(heuristic_mean)) * 100
    print(f"Improvement over Heuristic: {improvement:+.1f}%")

    # Plot training progress using enhanced visualization
    logger.plot_progress()

    # Success criteria
    if improvement > 15.0:
        print(f"\n🎯 SUCCESS: System is Learning effectively!")
        print(f" The implementation shows {improvement:.1f}% improvement over Heuristic baseline.")
    elif improvement > 5.0:
        print(f"\n✅ STABLE: System is Learning but could be improved.")
        print(f" Current improvement: {improvement:.1f}% over Heuristic.")
    else:
        print(f"\n⚠️ NEEDS TUNING: System performance needs improvement.")
        print(f" Consider hyperparameter tuning or longer training.")

    print(f"\n💾 Models saved to: /content/checkpoints/")

    # Save training data for future analysis
    training_data_path = "/content/training_data.npy"
    np.save(training_data_path, logger.episode_data)
    print(f"📊 Training data saved to: {training_data_path}")

    return {
        'final_eval': final_eval if 'final_eval' in locals() else None,
        'improvement': improvement,
        'training_data': logger.episode_data
    }
# output_multiplier
# Add to main execution block
if __name__ == "__main__":
    # Configure for effective training
    config.config.training.TOTAL_TIMESTEPS = 50000
    config.config.training.UPDATE_TIMESTEPS = 512
    config.config.colab.DEBUG_MODE = False

    set_seed(8723822)

    # Check if we should run training or just evaluation
    import sys
    if len(sys.argv) > 1 and sys.argv[1] == "evaluate":
        # Run only evaluation
        evaluate_trained_models()
    else:
        # Run full training
        results = main_training()